# F1 Podium Prediction — Milestone 1
**Pipeline:** audit → cleaning → data-engineering questions → leakage-safe features → modeling (LR / RF / GBM / shallow NN) → ablation → preprocessing-order experiments → XAI → inference function → report.

**Unit of analysis:** one driver entering one race. **Target:** `podium = positionOrder <= 3`. **Prediction cutoff:** *after qualifying, before the race starts*.

> **Run-all ready.** Put the Kaggle CSVs (14 files) in `./data/` (or set env var `F1_DATA_DIR`). If not found, the notebook tries `kagglehub`.

**Sections**
0. Setup  1. Raw audit (EDA before)  2. Cleaning  3. Keys & duplicates  4. EDA after cleaning  5. Data-engineering questions  6. Feature engineering (leakage-safe)  7. Feature selection evidence  8. Split & preprocessing  9. Models  10. Ablation  11. Preprocessing-order experiments  12. XAI  13. Inference function  14. Report

> **Revision note.** Re-run top-to-bottom before submitting: all saved outputs were cleared because several cells changed (Q1/Q2 conclusions, home-country mapping, evidence-based feature pruning, four-model ablation incl. RF with grid/qualifying split, data-driven XAI interpretation, more robust inference, richer report).


## 0. Setup

In [ ]:
import os, sys, re, json, hashlib, warnings, subprocess, importlib, time, math
from pathlib import Path
def _ensure(pkg, imp=None):
    try: importlib.import_module(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
for p,i in [('pandas',None),('numpy',None),('matplotlib',None),('seaborn',None),('scikit-learn','sklearn'),
            ('scipy',None),('statsmodels',None),('shap',None),('lime',None),('torch',None)]:
    _ensure(p,i)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython import get_ipython
from IPython.display import display as _ipy_display, Markdown
IN_IPY=get_ipython() is not None
if IN_IPY:
    try: get_ipython().run_line_magic('matplotlib','inline')
    except Exception: pass
else:
    import matplotlib; matplotlib.use('Agg')
def display(*objs):
    # works in Jupyter/Kaggle AND in script mode (falls back to print)
    for o in objs:
        if IN_IPY: _ipy_display(o)
        elif isinstance(o,Markdown): print('\n'+o.data+'\n')
        elif hasattr(o,'to_string'): print(o.to_string())
        else: print(o)
RESULTS_DIR=Path('results'); RESULTS_DIR.mkdir(exist_ok=True)
warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
FIG=Path('figures'); FIG.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi':110,'axes.titleweight':'bold'})
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/f'{name}.png', dpi=130, bbox_inches='tight')
    if IN_IPY: plt.show()
    else: plt.close('all')
    print(f'[figure saved] figures/{name}.png')
ANSWERS={}   # text answers collected for the final report
print('pandas',pd.__version__)

### Locate the data
Looks in `$F1_DATA_DIR`, `./data`, `./f1_data`, `.`, `/mnt/user-data/uploads`; otherwise tries `kagglehub` (needs internet + Kaggle access).

In [ ]:
CSV_NAMES=['circuits','constructors','constructor_results','constructor_standings','drivers','driver_standings',
           'races','results','qualifying','lap_times','pit_stops','sprint_results','seasons','status']
def find_data_dir():
    cands=[os.environ.get('F1_DATA_DIR'),'/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020','/kaggle/input/formula-1-world-championship-1950-2020',
           'data','f1_data','.','/mnt/user-data/uploads']
    for c in cands:
        if c and Path(c).is_dir() and (Path(c)/'results.csv').exists(): return Path(c)
    for root in ['/kaggle/input','.']:                      # last local resort: search for results.csv anywhere under these roots
        if Path(root).is_dir():
            hits=[p.parent for p in Path(root).rglob('results.csv') if (p.parent/'races.csv').exists()]
            if hits: return hits[0]
    try:
        _ensure('kagglehub'); import kagglehub
        return Path(kagglehub.dataset_download('rohanrao/formula-1-world-championship-1950-2020'))
    except Exception as e:
        raise FileNotFoundError('F1 CSVs not found. Put the 14 CSV files in ./data or set F1_DATA_DIR. '+str(e))
DATA_DIR=find_data_dir(); print('Data dir:',DATA_DIR.resolve())
missing=[n for n in CSV_NAMES if not (DATA_DIR/f'{n}.csv').exists()]
assert not missing, f'Missing CSVs: {missing}'
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1<<20),b''): h.update(ch)
    return h.hexdigest()
HASH_BEFORE={n:sha(DATA_DIR/f'{n}.csv') for n in CSV_NAMES}   # proves raw files stay immutable
# Raw load as strings, NO NA conversion: the sentinel r'\N' stays visible for the audit
RAW={n:pd.read_csv(DATA_DIR/f'{n}.csv',dtype=str,keep_default_na=False,na_values=[]) for n in CSV_NAMES}
display(pd.DataFrame({'rows':{n:len(d) for n,d in RAW.items()},'cols':{n:d.shape[1] for n,d in RAW.items()}}))

## 1. Raw audit — EDA *before* cleaning

In [ ]:
rows=[]
for n,d in RAW.items():
    for c in d.columns:
        k=int((d[c]==r'\N').sum()); e=int((d[c]=='').sum())
        if k or e: rows.append((n,c,k,e,100*(k+e)/len(d)))
SENT=pd.DataFrame(rows,columns=['table','column','n_sentinel_\\N','n_empty','pct_missing']).sort_values('pct_missing',ascending=False)
display(SENT.head(25))
top=SENT.head(25).assign(label=lambda x:x.table+'.'+x.column)
plt.figure(figsize=(9,7)); sns.barplot(data=top,y='label',x='pct_missing',color='#d1495b')
plt.title(r'BEFORE cleaning: share of sentinel (\N) cells — top 25 columns'); plt.xlabel('% of rows'); plt.ylabel('')
savefig('01_sentinels_before')
print('Total \\N cells across all tables:',int(SENT['n_sentinel_\\N'].sum()))

In [ ]:
# Raw dtype problem: everything is 'str' -> numeric ops silently fail / sort lexicographically
r=RAW['results']
print('results.position raw values containing \\N:',(r.position==r'\N').sum(),'of',len(r))
print('Lexicographic max of raw "grid" (string sort!):',r.grid.max(),'| numeric max:',pd.to_numeric(r.grid).max())
q=RAW['qualifying']; print('Raw q1 examples:',q.q1[q.q1!=r'\N'].head(3).tolist())

## 2. Cleaning (on copies — raw frames in `RAW` and raw CSV files are never modified)
Rules:
* `\N` and empty string → `NaN`.
* Columns that are fully numeric after removing missing → numeric (int if no missing & integer-valued).
* Date columns → `datetime64`; clock-time columns → `timedelta`.
* Lap-time strings (`m:ss.mmm`) → numeric seconds in new `*_s` columns (strings kept for traceability).

In [ ]:
DATE_COLS={'races':['date','fp1_date','fp2_date','fp3_date','quali_date','sprint_date'],'drivers':['dob']}
CLOCK_COLS={'races':['time','fp1_time','fp2_time','fp3_time','quali_time','sprint_time'],'pit_stops':['time']}
LAPTIME_COLS={'qualifying':['q1','q2','q3'],'results':['fastestLapTime'],'lap_times':['time'],'pit_stops':['duration']}
def to_seconds(s):
    if s is None or (isinstance(s,float) and np.isnan(s)): return np.nan
    try: parts=[float(p) for p in str(s).strip().split(':')]
    except ValueError: return np.nan
    sec=0.0
    for p in parts: sec=sec*60+p
    return sec
def clean_table(name,df):
    d=df.copy()
    d=d.mask(d.isin([r'\N','']))
    for c in DATE_COLS.get(name,[]):  d[c]=pd.to_datetime(d[c],errors='coerce')
    for c in CLOCK_COLS.get(name,[]): d[c]=pd.to_timedelta(d[c],errors='coerce')
    for c in LAPTIME_COLS.get(name,[]): d[c+'_s']=d[c].map(to_seconds).astype(float)
    skip=set(DATE_COLS.get(name,[])+CLOCK_COLS.get(name,[]))
    for c in d.columns:
        if c in skip or d[c].dtype.kind in 'fiM m'.replace(' ',''): continue
        num=pd.to_numeric(d[c],errors='coerce')
        if d[c].notna().any() and (num.notna()==d[c].notna()).all():
            d[c]=num.astype('int64') if (num.notna().all() and (num%1==0).all()) else num.astype(float)
    return d
T={n:clean_table(n,RAW[n]) for n in CSV_NAMES}
assert all(sha(DATA_DIR/f'{n}.csv')==HASH_BEFORE[n] for n in CSV_NAMES); print('Raw CSV files unchanged (sha256 verified).')
display(T['results'].dtypes.to_frame('dtype_after').T)
display(T['qualifying'][['q1','q1_s','q2_s','q3_s']].dropna().head())

In [ ]:
# AFTER-cleaning missingness (true NaN) + effect of lap-time conversion
miss_after=pd.DataFrame([(n,c,100*T[n][c].isna().mean()) for n in CSV_NAMES for c in T[n].columns],columns=['table','column','pct_nan'])
miss_after=miss_after[miss_after.pct_nan>0].sort_values('pct_nan',ascending=False).head(25)
fig,ax=plt.subplots(1,2,figsize=(14,6))
sns.barplot(data=top,y='label',x='pct_missing',color='#d1495b',ax=ax[0]); ax[0].set_title(r'BEFORE: \N strings'); ax[0].set_ylabel('')
sns.barplot(data=miss_after.assign(label=lambda x:x.table+'.'+x.column),y='label',x='pct_nan',color='#2e86ab',ax=ax[1]); ax[1].set_title('AFTER: real NaN (typed)'); ax[1].set_ylabel('')
savefig('02_missing_before_after')
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].hist(T['qualifying'].q1.dropna().str.len(),bins=15,color='grey'); ax[0].set_title('q1 as strings: just text lengths')
ax[1].hist(T['qualifying'].q1_s.dropna().clip(40,140),bins=60,color='#2e86ab'); ax[1].set_title('q1_s in seconds (numeric)'); ax[1].set_xlabel('seconds')
savefig('03_laptime_semantic_typing')

## 3. Primary-key / foreign-key verification and duplicate (raceId, driverId) handling

In [ ]:
PK={'circuits':['circuitId'],'constructors':['constructorId'],'constructor_results':['constructorResultsId'],
 'constructor_standings':['constructorStandingsId'],'drivers':['driverId'],'driver_standings':['driverStandingsId'],
 'lap_times':['raceId','driverId','lap'],'pit_stops':['raceId','driverId','stop'],'qualifying':['qualifyId'],
 'races':['raceId'],'results':['resultId'],'seasons':['year'],'sprint_results':['resultId'],'status':['statusId']}
pk_rows=[]
for n,k in PK.items():
    d=T[n]; pk_rows.append((n,'+'.join(k),int(d[k].isna().any(axis=1).sum()),int(d.duplicated(k).sum()),len(d)))
PKR=pd.DataFrame(pk_rows,columns=['table','primary_key','null_rows','duplicate_rows','n_rows']); display(PKR)
FK=[('results','raceId','races','raceId'),('results','driverId','drivers','driverId'),('results','constructorId','constructors','constructorId'),('results','statusId','status','statusId'),
 ('qualifying','raceId','races','raceId'),('qualifying','driverId','drivers','driverId'),('qualifying','constructorId','constructors','constructorId'),
 ('races','circuitId','circuits','circuitId'),('races','year','seasons','year'),
 ('driver_standings','raceId','races','raceId'),('driver_standings','driverId','drivers','driverId'),
 ('constructor_standings','raceId','races','raceId'),('constructor_standings','constructorId','constructors','constructorId'),
 ('constructor_results','raceId','races','raceId'),('constructor_results','constructorId','constructors','constructorId'),
 ('lap_times','raceId','races','raceId'),('lap_times','driverId','drivers','driverId'),
 ('pit_stops','raceId','races','raceId'),('pit_stops','driverId','drivers','driverId'),
 ('sprint_results','raceId','races','raceId'),('sprint_results','driverId','drivers','driverId'),('sprint_results','constructorId','constructors','constructorId'),('sprint_results','statusId','status','statusId')]
fk_rows=[]
for ct,cc,pt,pc in FK:
    vals=T[ct][cc].dropna(); orphan=int((~vals.isin(set(T[pt][pc]))).sum())
    fk_rows.append((f'{ct}.{cc}',f'{pt}.{pc}',len(vals),orphan,100*orphan/max(len(vals),1)))
FKR=pd.DataFrame(fk_rows,columns=['child','parent','n_checked','orphans','pct_orphan']); display(FKR)
plt.figure(figsize=(9,6)); sns.barplot(data=FKR.assign(l=FKR.child+' → '+FKR.parent),y='l',x='orphans',color='#6a994e')
plt.title('Foreign-key orphans per relationship (0 = reconciles)'); plt.ylabel('')
savefig('04_fk_orphans')

In [ ]:
# Duplicate (raceId, driverId) in results
res=T['results'].merge(T['races'][['raceId','year']],on='raceId',how='left')
dmask=res.duplicated(['raceId','driverId'],keep=False)
dups=res[dmask]
print(f'Rows in a duplicated (raceId,driverId) group: {dmask.sum()} | extra rows to drop: {res.duplicated(["raceId","driverId"]).sum()}')
dec=(dups.year//10*10).value_counts().sort_index()
plt.figure(figsize=(7,3.5)); dec.plot.bar(color='#d1495b'); plt.title('Duplicate (raceId, driverId) rows by decade (shared drives)'); plt.ylabel('rows')
savefig('05_duplicates_by_decade')
display(dups.sort_values(['raceId','driverId']).head(10)[['raceId','driverId','constructorId','grid','positionOrder','points','statusId','year']])

### Duplicate rule (and why)
Rows for the same driver in the same race are shared drives (car swaps). **Rule:** prefer the row with `grid > 0` (the car the driver actually qualified/started in — the only one a post-qualifying prediction could refer to), then the **best `positionOrder`** (a driver who shared a podium car did stand on the podium), then lowest `resultId`.  
Sensitivity: we count how many target labels would change under two alternative rules.

In [ ]:
def dedupe_results(res_df, rule='grid_then_best'):
    d=res_df.copy(); d['_hg']=(d.grid>0).astype(int)
    if rule=='grid_then_best': d=d.sort_values(['raceId','driverId','_hg','positionOrder','resultId'],ascending=[True,True,False,True,True])
    elif rule=='best_only':    d=d.sort_values(['raceId','driverId','positionOrder','resultId'])
    elif rule=='worst_only':   d=d.sort_values(['raceId','driverId','positionOrder','resultId'],ascending=[True,True,False,True])
    elif rule=='first_resultId': d=d.sort_values(['raceId','driverId','resultId'])
    return d.drop_duplicates(['raceId','driverId'],keep='first').drop(columns='_hg')
R_main=dedupe_results(T['results'])
chk=R_main[['raceId','driverId','positionOrder']].assign(p=lambda x:x.positionOrder<=3)
sens=[]
for rule in ['best_only','worst_only','first_resultId']:
    alt=dedupe_results(T['results'],rule)[['raceId','driverId','positionOrder']].assign(p2=lambda x:x.positionOrder<=3)
    m=chk.merge(alt[['raceId','driverId','p2']],on=['raceId','driverId']); sens.append((rule,int((m.p!=m.p2).sum())))
print('Rows before:',len(T['results']),'after:',len(R_main)); display(pd.DataFrame(sens,columns=['alt_rule','labels_that_differ_from_main']))
assert not R_main.duplicated(['raceId','driverId']).any()
# qualifying duplicates, if any: keep best position
Q=T['qualifying'].sort_values(['raceId','driverId','position']).drop_duplicates(['raceId','driverId'],keep='first')
print('Qualifying rows dropped as duplicates:',len(T['qualifying'])-len(Q))
# standings uniqueness on (raceId, driverId / constructorId)
DS=T['driver_standings'].drop_duplicates(['raceId','driverId']); CS=T['constructor_standings'].drop_duplicates(['raceId','constructorId'])
ANSWERS['dups']=f"{len(T['results'])-len(R_main)} duplicate result rows removed (shared drives); labels changing under alt rules: {dict(sens)}"

## 4. EDA after cleaning (impact of cleaning)

In [ ]:
RACES=T['races'].sort_values(['year','round']).reset_index(drop=True); RACES['race_order']=np.arange(len(RACES))
E=R_main.merge(RACES[['raceId','year','round']],on='raceId'); E['podium']=(E.positionOrder<=3).astype(int)
yr=E.groupby('year').agg(entries=('driverId','size'),podium_rate=('podium','mean'))
qy=Q.merge(RACES[['raceId','year']],on='raceId').groupby('year').size().reindex(yr.index).fillna(0)
yr['quali_coverage']=(qy/yr.entries).clip(upper=1)
fig,ax=plt.subplots(1,3,figsize=(17,4))
yr.entries.plot(ax=ax[0],color='#2e86ab'); ax[0].set_title('Entries per season (after dedupe)')
yr.podium_rate.plot(ax=ax[1],color='#d1495b'); ax[1].set_title('Podium rate per season (≈3/field size)')
yr.quali_coverage.plot(ax=ax[2],color='#6a994e'); ax[2].set_title('Qualifying coverage per season'); ax[2].axhline(.9,ls='--',c='k')
savefig('06_eda_after_cleaning')
fig,ax=plt.subplots(1,2,figsize=(12,4))
pre=T['results'].positionOrder; ax[0].hist(pre,bins=range(1,40),color='grey'); ax[0].set_title('positionOrder BEFORE dedupe')
ax[1].hist(R_main.positionOrder,bins=range(1,40),color='#2e86ab'); ax[1].set_title('positionOrder AFTER dedupe')
savefig('07_positionorder_before_after')
print(f'Podium prevalence overall: {E.podium.mean():.3f}')
# first year after which qualifying coverage stays >= 90%  -> modeling start year (data-driven)
cov=yr.quali_coverage; ok=[y for y in cov.index if (cov.loc[y:]>=0.9).all()]
START_YEAR=int(min(ok)); print('Modeling start year (qualifying coverage >= 90% from here on):',START_YEAR)
ANSWERS['start_year']=START_YEAR

In [ ]:
# Extra EDA on the cleaned, deduplicated table: the target, its main driver (grid slot) and DNF drift
EA=R_main.merge(RACES[['raceId','year']],on='raceId').merge(T['status'],on='statusId',how='left')
EA['podium']=(EA.positionOrder<=3).astype(int); EA['dnf']=1-((EA.status=='Finished')|EA.status.fillna('').str.startswith('+')).astype(int)
fig,ax=plt.subplots(1,3,figsize=(17,4))
g=EA[(EA.year>=START_YEAR)&(EA.grid>=1)&(EA.grid<=24)].groupby('grid').podium.mean(); ax[0].bar(g.index,g.values,color='#2e86ab'); ax[0].set_title(f'Podium rate by grid slot ({START_YEAR}+)'); ax[0].set_xlabel('grid slot'); ax[0].set_ylabel('podium rate')
d=EA.groupby('year').dnf.mean(); ax[1].plot(d.index,d.values,color='#d1495b'); ax[1].set_title('DNF share per season (reliability drift)'); ax[1].set_xlabel('year')
e=EA[EA.year>=START_YEAR].assign(era=lambda x:np.where(x.year>=2014,'hybrid 2014+','2003-2013')).groupby(['era','grid']).podium.mean().unstack(0).loc[1:12]; e.plot(ax=ax[2],marker='o'); ax[2].set_title('Podium rate by grid slot, by era'); ax[2].set_xlabel('grid slot')
savefig('06b_eda_target')
print(f'Target prevalence {START_YEAR}+: {EA[EA.year>=START_YEAR].podium.mean():.3f}; pole converts to a podium {EA[(EA.year>=START_YEAR)&(EA.grid==1)].podium.mean():.0%} of the time -> grid is expected to dominate (tested in §8).')


## 5. Base table: one row per (raceId, driverId), multi-hop joins

Joins: `results` (deduped) ⟶ `races` (year, round, date, circuit) ⟶ `circuits` (country) ⟶ `drivers` (nationality, dob) ⟶ `status` ⟶ `qualifying` (position, Q1–Q3 seconds).  
All joins are validated (`many_to_one` / `one_to_one`) so the grain cannot silently explode.

In [ ]:
Qf=Q.copy()
Qf['best_s']=Qf[['q1_s','q2_s','q3_s']].min(axis=1)
Qf['q1_gap_pct']=(Qf.q1_s/Qf.groupby('raceId').q1_s.transform('min')-1)*100
Qf['best_gap_pct']=(Qf.best_s/Qf.groupby('raceId').best_s.transform('min')-1)*100
Qf=Qf.rename(columns={'position':'quali_position'})[['raceId','driverId','quali_position','q1_gap_pct','best_gap_pct']]

def make_base():
    b=R_main.merge(RACES[['raceId','year','round','circuitId','date','race_order']],on='raceId',how='left',validate='m:1')
    b=b.merge(T['circuits'][['circuitId','country']].rename(columns={'country':'circuit_country'}),on='circuitId',how='left',validate='m:1')
    # FIX: the circuits table uses BOTH 'USA' and 'United States' -> harmonise, otherwise US drivers at 'United States' circuits are never 'home'
    b['circuit_country']=b.circuit_country.str.strip().replace({'United States':'USA','United Kingdom':'UK','United States of America':'USA'})
    b=b.merge(T['drivers'][['driverId','dob','nationality']],on='driverId',how='left',validate='m:1')
    b['nationality']=b.nationality.str.strip()    # FIX: 'Argentinian ' had a trailing space and was silently unmapped
    b=b.merge(T['status'],on='statusId',how='left',validate='m:1')
    b=b.merge(Qf,on=['raceId','driverId'],how='left',validate='1:1')
    b['podium']=(b.positionOrder<=3).astype(int)
    b['finished']=((b.status=='Finished')|b.status.fillna('').str.startswith('+')).astype(int)
    b['field_size']=b.groupby('raceId').driverId.transform('count')
    return b.sort_values(['race_order','driverId']).reset_index(drop=True)
BASE=make_base()
assert not BASE.duplicated(['raceId','driverId']).any() and len(BASE)==len(R_main)
print(BASE.shape); display(BASE.head(3))

### Q1 — Which circuits convert a front-row start into a podium most often; does the ranking change after 2014?
* **Grain:** one driver-entry in one race. **Front row** is defined two ways: `quali_position ≤ 2` (qualifying result) and `grid ≤ 2` (starting grid after penalties). They differ when penalties apply.
* **Joins:** results → races → circuits → qualifying.
* **Missingness:** `quali_position` is missing outside the qualifying-covered era, so both definitions are compared on the same era window (pre-hybrid = START_YEAR–2013, hybrid = 2014+).
* **Cutoff:** both are known before the start. We minimum-sample filter and show Wilson 95% CIs because per-circuit samples are small.

In [ ]:
from scipy import stats
def wilson(k,n,z=1.96):
    if n==0: return (np.nan,np.nan)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); a=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return ((c-a)/d,(c+a)/d)
W=BASE[BASE.year>=START_YEAR].copy()
W['era']=np.where(W.year>=2014,'hybrid (2014+)',f'pre-hybrid ({START_YEAR}-2013)')
W=W.merge(T['circuits'][['circuitId','name']].rename(columns={'name':'circuit'}),on='circuitId',how='left')
print('Entries with grid!=quali position:',int(((W.grid!=W.quali_position)&W.quali_position.notna()&(W.grid>0)).sum()),'of',int(W.quali_position.notna().sum()))
def conv(df,col,thr=2):
    f=df[(df[col]>=1)&(df[col]<=thr)]
    g=f.groupby('circuit').agg(front_row_starts=('podium','size'),podiums=('podium','sum')); g['rate']=g.podiums/g.front_row_starts
    lo_hi=g.apply(lambda r:wilson(r.podiums,r.front_row_starts),axis=1); g['lo']=[x[0] for x in lo_hi]; g['hi']=[x[1] for x in lo_hi]
    return g
MIN_ALL,MIN_ERA=30,10
out={}
for col in ['quali_position','grid']:
    allg=conv(W,col); allg=allg[allg.front_row_starts>=MIN_ALL].sort_values('rate',ascending=False)
    pre=conv(W[W.era.str.startswith('pre')],col); hyb=conv(W[W.era.str.startswith('hybrid')],col)
    both=pre.join(hyb,lsuffix='_pre',rsuffix='_hyb',how='inner'); both=both[(both.front_row_starts_pre>=MIN_ERA)&(both.front_row_starts_hyb>=MIN_ERA)]
    both['rank_pre']=both.rate_pre.rank(ascending=False); both['rank_hyb']=both.rate_hyb.rank(ascending=False)
    rho,p=stats.spearmanr(both.rank_pre,both.rank_hyb) if len(both)>3 else (np.nan,np.nan)
    out[col]=dict(all=allg,both=both,rho=rho,p=p)
    print(f'[{col}] circuits (all-era, ≥{MIN_ALL} front-row starts): {len(allg)} | circuits in both eras (≥{MIN_ERA}): {len(both)} | Spearman rank corr pre↔hybrid = {rho:.2f} (p={p:.3f})')
fig,ax=plt.subplots(1,2,figsize=(16,7))
for a,col in zip(ax,['quali_position','grid']):
    g=out[col]['all'].head(15).iloc[::-1]
    a.barh(g.index,g.rate,xerr=[g.rate-g.lo,g.hi-g.rate],color='#2e86ab',alpha=.85); a.set_xlim(0,1)
    a.set_title(f'Top-15 front-row→podium conversion\n(front row = {col} ≤ 2)'); a.set_xlabel('podium rate')
savefig('08_q1_top_circuits')
fig,ax=plt.subplots(1,2,figsize=(14,6))
for a,col in zip(ax,['quali_position','grid']):
    b=out[col]['both']
    a.scatter(b.rate_pre,b.rate_hyb,s=b.front_row_starts_hyb*3,alpha=.6); a.plot([0,1],[0,1],'k--',lw=1)
    for n_,r in b.iterrows():
        if abs(r.rate_hyb-r.rate_pre)>.25: a.annotate(n_[:18],(r.rate_pre,r.rate_hyb),fontsize=7)
    a.set_xlim(0,1.02); a.set_ylim(0,1.02); a.set_title(f'{col}: pre-hybrid vs hybrid\nSpearman ρ={out[col]["rho"]:.2f}'); a.set_xlabel('pre-hybrid rate'); a.set_ylabel('hybrid rate')
savefig('09_q1_era_change')
mv=out['quali_position']['both'].assign(change=lambda x:x.rank_hyb-x.rank_pre).sort_values('change')
display(mv[['front_row_starts_pre','rate_pre','front_row_starts_hyb','rate_hyb','rank_pre','rank_hyb','change']].round(2).head(8))
top_q=out['quali_position']['all'].head(3).index.tolist(); top_g=out['grid']['all'].head(3).index.tolist()
def _rho_txt(r,p):
    if np.isnan(r): return 'not estimable'
    lvl='strong' if r>=.7 else 'moderate' if r>=.4 else 'weak'
    return f"{lvl} (Spearman ρ={r:.2f}, p={p:.3f}{'; not significant' if p>=.05 else ''})"
rq,pq=out['quali_position']['rho'],out['quali_position']['p']; rg,pg=out['grid']['rho'],out['grid']['p']
pooled=lambda col,e:(lambda f:(f.podium.sum()/len(f)))(W[(W[col]>=1)&(W[col]<=2)&W.era.str.startswith(e)])
pr_q=(pooled('quali_position','pre'),pooled('quali_position','hybrid')); pr_g=(pooled('grid','pre'),pooled('grid','hybrid'))
if max(rq,rg)<0.5: verdict="The circuit ranking therefore changes substantially after 2014 under both definitions."
elif min(rq,rg)<0.5: verdict=("The ranking is only partly stable: it is "+('moderately preserved' if rq>=0.5 else 'weakly preserved')+" when the front row is defined by qualifying position but "+('moderately preserved' if rg>=0.5 else 'weakly preserved')+" when defined by the penalty-adjusted grid slot, so penalties matter for which circuits look 'good at converting'.")
else: verdict="The circuit ranking is moderately/strongly preserved after 2014 under both definitions."
ANSWERS['Q1']=(f"Top front-row→podium circuits (≥{MIN_ALL} front-row starts) — by qualifying position: {top_q}; by grid slot: {top_g}. "
 f"Pooled front-row conversion rises from {pr_q[0]:.0%} to {pr_q[1]:.0%} (qualifying definition) and {pr_g[0]:.0%} to {pr_g[1]:.0%} (grid definition) in the hybrid era. "
 f"Rank agreement pre-hybrid vs hybrid: qualifying definition {_rho_txt(rq,pq)}; grid definition {_rho_txt(rg,pg)}. {verdict} Per-circuit samples are small (see Wilson CIs).")
display(Markdown('**Answer Q1.** '+ANSWERS['Q1']))

### Q2 — Do drivers podium more at home circuits, controlling for grid position?
* **Grain:** driver-entry. **Home** = driver nationality maps to the circuit's country (`drivers.nationality` ⟶ country via a documented demonym table; unmapped nationalities are reported).
* **Control:** grid slot (pit-lane starts excluded). We compare drivers from similar grid slots via (a) stratified rates, (b) Mantel–Haenszel common odds ratio over grid strata, (c) logistic regressions with grid fixed effects (+ decade fixed effects), SEs clustered by race.
* **Cutoff:** home flag and grid are pre-race. **Caveat:** constructor strength is not controlled (e.g. Italians at Ferrari at Monza).

In [ ]:
NAT2COUNTRY={'American':'USA','American-Italian':'USA','Argentine':'Argentina','Argentine-Italian':'Argentina','Argentinian':'Argentina','Australian':'Australia','Austrian':'Austria',
 'Belgian':'Belgium','Brazilian':'Brazil','British':'UK','Canadian':'Canada','Chilean':'Chile','Chinese':'China','Colombian':'Colombia','Czech':'Czech Republic','Danish':'Denmark',
 'Dutch':'Netherlands','East German':'Germany','Emirati':'UAE','Finnish':'Finland','French':'France','German':'Germany','Hong Kong':'China','Hungarian':'Hungary','Indian':'India','Indonesian':'Indonesia',
 'Irish':'Ireland','Italian':'Italy','Japanese':'Japan','Liechtensteiner':'Switzerland','Malaysian':'Malaysia','Mexican':'Mexico','Monegasque':'Monaco','New Zealander':'New Zealand','Polish':'Poland',
 'Portuguese':'Portugal','Rhodesian':'Rhodesia','Russian':'Russia','South African':'South Africa','Spanish':'Spain','Swedish':'Sweden','Swiss':'Switzerland','Thai':'Thailand','Uruguayan':'Uruguay','Venezuelan':'Venezuela'}
BASE['driver_country']=BASE.nationality.map(NAT2COUNTRY)
BASE['home_race']=(BASE.driver_country==BASE.circuit_country).astype(int)
print('Circuit countries in data:',sorted(BASE.circuit_country.dropna().unique())[:40])
unm=sorted(set(BASE.nationality.dropna())-set(NAT2COUNTRY)); print('Unmapped nationalities (treated as non-home):',unm)
assert not unm, f'unmapped nationalities: {unm}'
mis=sorted(set(NAT2COUNTRY.values())-set(BASE.circuit_country.dropna().unique())); print('Mapped driver countries with no circuit (never home):',mis)
print('Sanity: American drivers flagged home at US circuits:',int(((BASE.nationality=='American')&(BASE.circuit_country=='USA')&(BASE.home_race==1)).sum()),'of',int(((BASE.nationality=='American')&(BASE.circuit_country=='USA')).sum()))
H=BASE[(BASE.grid>=1)&BASE.driver_country.notna()].copy()
H['grid_bin']=pd.cut(H.grid,[0,1,2,3,4,5,6,8,10,13,16,40],labels=['1','2','3','4','5','6','7-8','9-10','11-13','14-16','17+'])
print(f'Home entries: {H.home_race.sum()} of {len(H)} ({100*H.home_race.mean():.1f}%)')

In [ ]:
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import StratifiedTable
H['decade']=(H.year//10*10).astype(str)
strat=H.groupby(['grid_bin','home_race']).podium.agg(['mean','size']).unstack('home_race')
fig,ax=plt.subplots(1,2,figsize=(14,4.5))
x=np.arange(len(strat)); ax[0].bar(x-.2,strat['mean'][0],.4,label='away'); ax[0].bar(x+.2,strat['mean'][1],.4,label='home')
ax[0].set_xticks(x); ax[0].set_xticklabels(strat.index); ax[0].set_title('Podium rate by grid bin: home vs away'); ax[0].set_xlabel('grid bin'); ax[0].legend()
mg=H.groupby('home_race').grid.mean(); ax[1].bar(['away','home'],mg.values,color=['#999','#d1495b']); ax[1].set_title('Mean grid slot (do home drivers start higher?)')
savefig('10_q2_home_vs_away')
tabs=[]
for gb,g in H.groupby('grid_bin'):
    t=pd.crosstab(g.home_race,g.podium).reindex(index=[1,0],columns=[1,0],fill_value=0)
    if t.values.sum()>0 and (t.loc[1].sum()>0): tabs.append(t.values)
st=StratifiedTable(tabs); mh=st.oddsratio_pooled; ci=st.oddsratio_pooled_confint()
print(f'Mantel–Haenszel pooled OR (home vs away, grid-stratified) = {mh:.2f}  95% CI [{ci[0]:.2f}, {ci[1]:.2f}]; homogeneity test p={st.test_equal_odds().pvalue:.3f}')
models={'A: home only':'podium ~ home_race','B: + grid FE':'podium ~ home_race + C(grid_bin)','C: + grid FE + decade FE':'podium ~ home_race + C(grid_bin) + C(decade)'}
rows=[]
for k,f in models.items():
    m=smf.logit(f,data=H).fit(disp=0,cov_type='cluster',cov_kwds={'groups':H.raceId})
    b=m.params['home_race']; lo,hi=m.conf_int().loc['home_race']; rows.append((k,np.exp(b),np.exp(lo),np.exp(hi),m.pvalues['home_race']))
Q2=pd.DataFrame(rows,columns=['model','OR','lo','hi','p']); Q2.loc[len(Q2)]=['Mantel–Haenszel (grid strata)',mh,ci[0],ci[1],np.nan]; display(Q2.round(3))
plt.figure(figsize=(8,3.5)); plt.errorbar(Q2.OR,range(len(Q2)),xerr=[Q2.OR-Q2.lo,Q2.hi-Q2.OR],fmt='o',capsize=4); plt.axvline(1,c='k',ls='--')
plt.yticks(range(len(Q2)),Q2.model); plt.xlabel('odds ratio for podium (home vs away)'); plt.title('Home advantage — odds ratio with 95% CI'); plt.gca().invert_yaxis()
savefig('11_q2_forest')
bB=Q2.iloc[1]; naive=Q2.OR[0]
start_txt=(f"home drivers start {'further back' if mg[1]>mg[0] else 'further forward'} on the grid on average (mean grid slot {mg[1]:.1f} vs {mg[0]:.1f} for away drivers)")
if bB.lo>1:   ctrl_txt="the interval lies above 1: home drivers podium MORE often than away drivers starting from the same grid slot (association only)"
elif bB.hi<1: ctrl_txt="the interval lies below 1: home drivers podium LESS often than away drivers starting from the same grid slot"
else:         ctrl_txt="the interval includes 1: no reliable home effect once starting position is held fixed"
gap_txt=("The unadjusted comparison is confounded by starting position: "+start_txt+". " if abs(np.log(naive))>abs(np.log(bB.OR)) else "Controlling for grid slot barely changes the estimate: "+start_txt+". ")
ANSWERS['Q2']=(f"Unadjusted, home drivers podium with OR={naive:.2f} (home {'less' if naive<1 else 'more'} often). {gap_txt}"
  f"After controlling for grid slot the OR is {bB.OR:.2f} (95% CI {bB.lo:.2f}–{bB.hi:.2f}; Mantel–Haenszel {mh:.2f}, adding decade fixed effects {Q2.OR[2]:.2f}); {ctrl_txt}. "
  "Caveats: this is an association, not a causal effect, and constructor strength is not controlled.")
display(Markdown('**Answer Q2.** '+ANSWERS['Q2']))

### Q3 — Constructors with the highest mechanical-retirement rate, 2014–2021 vs 2022–2024
* **Grain:** one race start (driver-entry) aggregated by constructor lineage. Rebrands are merged (Toro Rosso→AlphaTauri→RB; Force India→Racing Point→Aston Martin; Lotus F1/Renault→Alpine; Sauber/Alfa Romeo) so the two periods compare the same team.
* **Joins:** results → status → races (year) → constructors.
* **Definition:** *mechanical* = component-failure statuses (engine/power unit, gearbox, hydraulics, electrical, brakes, suspension, cooling…). **Excluded:** accidents, collisions, spins, disqualifications (regulation), withdrawals, generic "Retired", tyre/puncture/wing damage (usually contact or debris). A broader sensitivity definition adds tyre/puncture/wing/undertray.
* **Missingness:** `status` has no NaN after FK check. Rate = mechanical retirements / race starts; minimum 30 starts per period; Wilson CIs shown.

In [ ]:
MECH_KW=['engine','gearbox','transmission','clutch','hydraulic','electrical','electronic','suspension','brake','radiator','oil','overheat','water','fuel pressure','fuel system','fuel pump',
 'turbo','wheel nut','power unit','power loss','mechanical','exhaust','driveshaft','differential','throttle','battery','cooling','crankshaft','halfshaft','alternator','gear','ignition','pneumatic','technical','steering','track rod','vibration','mgu','spark','fluid','hybrid']
EXCL_KW=['accident','collision','spun','disqualif','withdr','retired','not qualify','not start','damage','seat','fire','illness','safety','excluded','puncture','tyre','wing','undertray','debris']
BROAD_EXTRA=['tyre','puncture','wing','undertray','wheel']
def is_mech(s,broad=False):
    s=str(s).lower()
    if broad and any(k in s for k in BROAD_EXTRA): return True
    if any(k in s for k in EXCL_KW) and not any(k in s for k in ['engine fire']): return False
    return any(k in s for k in MECH_KW) or s.strip()=='ers'
st_tab=T['status'].copy(); st_tab['mechanical_strict']=st_tab.status.map(is_mech); st_tab['mechanical_broad']=st_tab.status.map(lambda s:is_mech(s,True))
use=BASE[(BASE.year>=2014)].status.value_counts().rename('n_2014plus'); st_tab=st_tab.merge(use,left_on='status',right_index=True,how='inner').sort_values('n_2014plus',ascending=False)
print('Status classification audit (statuses seen 2014+):'); display(st_tab.head(40))
LINEAGE={'toro_rosso':'AlphaTauri/RB (Toro Rosso)','alphatauri':'AlphaTauri/RB (Toro Rosso)','rb':'AlphaTauri/RB (Toro Rosso)','force_india':'Aston Martin (Force India)','racing_point':'Aston Martin (Force India)','aston_martin':'Aston Martin (Force India)',
 'lotus_f1':'Alpine (Renault/Lotus)','renault':'Alpine (Renault/Lotus)','alpine':'Alpine (Renault/Lotus)','sauber':'Sauber/Alfa Romeo','alfa':'Sauber/Alfa Romeo'}
cons=T['constructors'].set_index('constructorId'); BASE['constructor']=BASE.constructorId.map(lambda i:LINEAGE.get(cons.constructorRef.get(i),cons.name.get(i)))
M=BASE.merge(st_tab[['status','mechanical_strict','mechanical_broad']],on='status',how='left'); M[['mechanical_strict','mechanical_broad']]=M[['mechanical_strict','mechanical_broad']].fillna(False)
M=M[(M.year>=2014)&~M.status.fillna('').str.contains('Did not|Withdrew')]   # non-starters are not race starts
M['period']=np.where(M.year<=2021,'2014–2021','2022–2024')
def mech_rate(df,col):
    g=df.groupby(['constructor','period']).agg(starts=('podium','size'),mech=(col,'sum')).reset_index(); g['rate']=g.mech/g.starts
    ci=g.apply(lambda r:wilson(r.mech,r.starts),axis=1); g['lo']=[c[0] for c in ci]; g['hi']=[c[1] for c in ci]; return g
G=mech_rate(M,'mechanical_strict'); MIN_STARTS=30
P=G.pivot(index='constructor',columns='period',values=['starts','mech','rate','lo','hi']); P=P[(P['starts']>=MIN_STARTS).all(axis=1)]
P['delta']=P['rate']['2022–2024']-P['rate']['2014–2021']; P=P.sort_values(('rate','2022–2024'),ascending=False)
fig,ax=plt.subplots(figsize=(10,6)); idx=np.arange(len(P)); h=.38
for j,(per,c) in enumerate(zip(['2014–2021','2022–2024'],['#2e86ab','#d1495b'])):
    r=P['rate'][per]; ax.barh(idx+(j-.5)*h,r,h,xerr=[r-P['lo'][per],P['hi'][per]-r],color=c,label=per,alpha=.9)
ax.set_yticks(idx); ax.set_yticklabels(P.index); ax.invert_yaxis(); ax.set_xlabel('mechanical retirements per race start'); ax.legend(); ax.set_title('Mechanical-retirement rate by constructor (strict definition, ≥30 starts/period)')
savefig('12_q3_mech_rate')
Gb=mech_rate(M,'mechanical_broad').pivot(index='constructor',columns='period',values='rate').reindex(P.index)
rho3=stats.spearmanr(P['rate']['2022–2024'],Gb['2022–2024'])[0]
tot=M.groupby('period').agg(s=('podium','size'),m=('mechanical_strict','sum')); ztest=stats.chi2_contingency([[tot.m['2014–2021'],tot.s['2014–2021']-tot.m['2014–2021']],[tot.m['2022–2024'],tot.s['2022–2024']-tot.m['2022–2024']]])
display(P[['starts','mech','rate','delta']].round(3))
t1=P['rate']['2014–2021'].idxmax(); t2=P['rate']['2022–2024'].idxmax()
ANSWERS['Q3']=(f"Highest mechanical-retirement rate 2014–2021: **{t1}** ({P['rate']['2014–2021'].max():.1%}); 2022–2024: **{t2}** ({P['rate']['2022–2024'].max():.1%}). "
 f"Field-wide rate moved from {tot.m['2014–2021']/tot.s['2014–2021']:.1%} to {tot.m['2022–2024']/tot.s['2022–2024']:.1%} (χ² p={'<0.001' if ztest[1]<0.001 else format(ztest[1],'.3f')}). Largest increase: {P['delta'].idxmax()} ({P['delta'].max():+.1%}); {'largest decrease' if P['delta'].min()<0 else 'smallest increase'}: {P['delta'].idxmin()} ({P['delta'].min():+.1%}). "
 f"2022–24 ranking is robust to a broader definition (Spearman {rho3:.2f}). Small samples per team: read with the CIs.")
display(Markdown('**Answer Q3.** '+ANSWERS['Q3']))

## 6. Feature engineering — leakage-safe, available *after qualifying and before the race*
**Banned (future) fields for the current race:** `positionOrder`, `position`, `points`, `laps`, `time`, `milliseconds`, `fastestLap*`, `statusId/status`, pit stops, lap times, post-race standings.

**Allowed feature groups (and why they are valid at the cutoff):**
| group | features | validity at cutoff |
|---|---|---|
| `grid_quali` | grid (filled), pit-lane flag, front-row flag, quali position, Q1/best-lap gap to pole (%), grid−quali (penalty), missing flag | qualifying is finished; grid incl. penalties is published before the start |
| `driver_form` | rolling mean of finishing position / podium / points-share / DNF / grid over the driver's **previous** 3–5 races, last position | `shift(1)` before rolling → only past races |
| `constructor_form` | same idea at constructor-race level, previous 5 races of that constructor | only past races |
| `standings` | driver & constructor points/position **before** this race (standings after the previous round of the same season) | uses previous round's `*_standings` only |
| `experience_context` | career starts/podiums (prior), age, round, field size | known pre-race |
| `circuit_history` | driver's prior starts/avg finish/podium rate at that circuit | prior editions only |
| `home` | driver nationality == circuit country | known pre-race |

Sprint results are *not* used (only exist on some weekends; listed under limitations).

In [ ]:
RACES['prev_raceId']=RACES.groupby('year').raceId.shift(1)
def build_features(base):
    b=base.sort_values(['race_order','driverId']).copy().reset_index(drop=True)
    b['race_pts_total']=b.groupby('raceId').points.transform('sum')
    b['points_share']=np.where(b.race_pts_total>0,b.points/b.race_pts_total,0.0)
    b['dnf']=1-b.finished
    b['is_pit_start']=(b.grid==0).astype(int)
    b['grid_filled']=np.where(b.grid==0,b.field_size,b.grid).astype(float)
    b['front_row']=(b.grid_filled<=2).astype(int)
    b['quali_missing']=b.quali_position.isna().astype(int)
    b['grid_minus_quali']=b.grid_filled-b.quali_position
    b['grid_penalty']=(b.grid_minus_quali>0).astype(int)
    b['age']=(b.date-b.dob).dt.days/365.25
    # ---- driver form (strictly previous races)
    b=b.sort_values(['driverId','race_order']); gd=b.groupby('driverId')
    roll=lambda col,n: gd[col].transform(lambda s:s.shift(1).rolling(n,min_periods=1).mean())
    b['form_pos_3']=roll('positionOrder',3); b['form_pos_5']=roll('positionOrder',5)
    b['form_podium_5']=roll('podium',5); b['form_points_5']=roll('points_share',5)
    b['form_dnf_5']=roll('dnf',5); b['form_grid_5']=roll('grid_filled',5)
    b['last_pos']=gd['positionOrder'].shift(1); b['no_history']=b.form_pos_5.isna().astype(int)
    b['career_starts']=gd.cumcount().astype(float)
    b['career_podiums']=gd['podium'].transform(lambda s:s.shift(1).cumsum()).fillna(0)
    # ---- circuit history
    gc=b.groupby(['driverId','circuitId'])
    b['circ_starts']=gc.cumcount().astype(float)
    b['circ_avg_pos']=gc['positionOrder'].transform(lambda s:s.shift(1).expanding().mean())
    b['circ_podium_rate']=gc['podium'].transform(lambda s:s.shift(1).expanding().mean())
    # ---- constructor form (strictly previous races of that constructor)
    cr=b.groupby(['raceId','constructorId']).agg(race_order=('race_order','first'),t_pos=('positionOrder','mean'),t_podium=('podium','mean'),t_pts=('points_share','sum')).reset_index()
    cr=cr.sort_values(['constructorId','race_order']); gcr=cr.groupby('constructorId')
    for src,dst in [('t_pos','team_form_pos_5'),('t_podium','team_form_podium_5'),('t_pts','team_form_points_5')]:
        cr[dst]=gcr[src].transform(lambda s:s.shift(1).rolling(5,min_periods=1).mean())
    b=b.merge(cr[['raceId','constructorId','team_form_pos_5','team_form_podium_5','team_form_points_5']],on=['raceId','constructorId'],how='left',validate='m:1')
    b['team_no_history']=b.team_form_pos_5.isna().astype(int)
    # ---- standings BEFORE this race (previous round of same season)
    b=b.merge(RACES[['raceId','prev_raceId']],on='raceId',how='left',validate='m:1')
    lead=DS.groupby('raceId').points.max().rename('drv_leader_pts').reset_index().rename(columns={'raceId':'prev_raceId'})
    ds=DS[['raceId','driverId','points','position','wins']].rename(columns={'raceId':'prev_raceId','points':'drv_points_pre','position':'drv_pos_pre','wins':'drv_wins_pre'})
    b=b.merge(ds,on=['prev_raceId','driverId'],how='left',validate='m:1').merge(lead,on='prev_raceId',how='left',validate='m:1')
    clead=CS.groupby('raceId').points.max().rename('con_leader_pts').reset_index().rename(columns={'raceId':'prev_raceId'})
    cs=CS[['raceId','constructorId','points','position']].rename(columns={'raceId':'prev_raceId','points':'con_points_pre','position':'con_pos_pre'})
    b=b.merge(cs,on=['prev_raceId','constructorId'],how='left',validate='m:1').merge(clead,on='prev_raceId',how='left',validate='m:1')
    b['standings_missing']=b.drv_points_pre.isna().astype(int)
    b['drv_pts_ratio']=np.where(b.drv_leader_pts>0,b.drv_points_pre/b.drv_leader_pts,0.0)
    b['drv_pts_ratio']=np.where(b.drv_points_pre.isna(),0.0,b.drv_pts_ratio)
    b['drv_pos_pre']=b.drv_pos_pre.fillna(b.field_size); b['drv_wins_pre']=b.drv_wins_pre.fillna(0.0)
    b['con_pts_ratio']=np.where(b.con_leader_pts>0,b.con_points_pre/b.con_leader_pts,np.nan)
    b['con_pos_pre']=b.con_pos_pre   # NaN where constructor standings do not exist (pre-1958 / round 1) -> imputed + flagged
    b['con_standings_missing']=b.con_pos_pre.isna().astype(int)
    return b.sort_values(['race_order','driverId']).reset_index(drop=True)

FEATURE_GROUPS={
 'grid_quali':['grid_filled','is_pit_start','front_row','quali_position','quali_missing','q1_gap_pct','best_gap_pct','grid_minus_quali','grid_penalty'],
 'driver_form':['form_pos_3','form_pos_5','form_podium_5','form_points_5','form_dnf_5','form_grid_5','last_pos','no_history'],
 'constructor_form':['team_form_pos_5','team_form_podium_5','team_form_points_5','team_no_history'],
 'standings':['drv_pts_ratio','drv_pos_pre','drv_wins_pre','con_pos_pre','con_pts_ratio','standings_missing','con_standings_missing'],
 'experience_context':['career_starts','career_podiums','age','round','field_size'],
 'circuit_history':['circ_starts','circ_avg_pos','circ_podium_rate'],
 'home':['home_race']}
FEATURES=[f for g in FEATURE_GROUPS.values() for f in g]
FEAT_GROUP_OF={f:g for g,fs in FEATURE_GROUPS.items() for f in fs}
BANNED={'positionOrder','position','positionText','points','laps','time','milliseconds','fastestLap','rank','fastestLapTime','fastestLapSpeed','statusId','status','finished','dnf','podium','points_share','race_pts_total'}
assert not (set(FEATURES)&BANNED), 'banned future column in feature list'
t0=time.time(); ALL=build_features(BASE); print(f'features built in {time.time()-t0:.1f}s ->',ALL.shape)
MODEL=ALL[ALL.year>=START_YEAR].reset_index(drop=True)   # engineer on full history, then filter (see EXP-C for the alternative)
print('Modeling table:',MODEL.shape,'| unique (raceId,driverId):',not MODEL.duplicated(['raceId','driverId']).any(),'| podium rate',round(MODEL.podium.mean(),3))

### 6.1 Leakage tests (executable evidence, not just a claim)

In [ ]:
rng=np.random.default_rng(SEED)
# TEST 1: manual recomputation of form_pos_5 / team form from raw past rows for random rows
sample=MODEL.sample(200,random_state=SEED); bad=0
for _,r in sample.iterrows():
    past=BASE[(BASE.driverId==r.driverId)&(BASE.race_order<r.race_order)].sort_values('race_order').tail(5)
    exp=past.positionOrder.mean() if len(past) else np.nan
    if not ((np.isnan(exp) and np.isnan(r.form_pos_5)) or abs(exp-r.form_pos_5)<1e-9): bad+=1
print(f'TEST 1  manual recompute of form_pos_5 on 200 rows: mismatches = {bad}'); assert bad==0
# TEST 2: perturb the CURRENT race outcome (positions, points, status) -> features of that race must NOT change
def perturb_race(base,rid):
    p=base.copy(); m=p.raceId==rid; n=m.sum()
    p.loc[m,'positionOrder']=rng.permutation(p.loc[m,'positionOrder'].values); p.loc[m,'points']=rng.permutation(p.loc[m,'points'].values)
    p.loc[m,'finished']=rng.integers(0,2,n); p['podium']=(p.positionOrder<=3).astype(int); return p
rids=rng.choice(MODEL.raceId.unique(),3,replace=False); ok_all=True
for rid in rids:
    f2=build_features(perturb_race(BASE,rid)); f2=f2[f2.year>=START_YEAR].reset_index(drop=True)
    a=MODEL[MODEL.raceId==rid].set_index('driverId')[FEATURES]; b2=f2[f2.raceId==rid].set_index('driverId')[FEATURES].loc[a.index]
    same=np.allclose(a.values.astype(float),b2.values.astype(float),equal_nan=True); ok_all&=same
    print(f'TEST 2  race {rid}: features unchanged after scrambling its own results -> {same}')
assert ok_all
# TEST 3: standings source race strictly precedes the current race
chk=MODEL[['raceId','prev_raceId']].dropna().merge(RACES[['raceId','race_order']],on='raceId').merge(RACES[['raceId','race_order']].rename(columns={'raceId':'prev_raceId','race_order':'prev_order'}),on='prev_raceId')
print('TEST 3  standings source strictly earlier than race:',bool((chk.prev_order<chk.race_order).all())); assert (chk.prev_order<chk.race_order).all()

## 7. Dataset split (train / validation / test) — chronological
**Choice:** the last 3 seasons = **test**, the 3 seasons before = **validation**, everything earlier (from the qualifying-covered start year) = **train**.  
**Why:** deployment predicts *future* races, so a random split would leak era, regulation and team-strength information across sets (shown empirically in EXP-D below). Validation chooses hyper-parameters and the F1 threshold; the test set is touched only for final reporting. Limitation: the test years form a distribution shift (e.g. 2022 ground-effect regulations).

In [ ]:
yrs=[int(y) for y in sorted(MODEL.year.unique())]; TEST_YEARS=yrs[-3:]; VAL_YEARS=yrs[-6:-3]
split=np.where(MODEL.year.isin(TEST_YEARS),'test',np.where(MODEL.year.isin(VAL_YEARS),'val','train')); MODEL['split']=split
tr,va,te=[MODEL[MODEL.split==s] for s in ['train','val','test']]
Xtr,ytr,Xva,yva,Xte,yte=tr[FEATURES],tr.podium.values,va[FEATURES],va.podium.values,te[FEATURES],te.podium.values
print(f'train {tr.year.min()}–{tr.year.max()}: {len(tr)} rows | val {VAL_YEARS}: {len(va)} | test {TEST_YEARS}: {len(te)}')
sp=MODEL.groupby('split').agg(rows=('podium','size'),podium_rate=('podium','mean')).loc[['train','val','test']]; display(sp)
fig,ax=plt.subplots(1,2,figsize=(12,3.5)); sp.rows.plot.bar(ax=ax[0],color=['#2e86ab','#f6ae2d','#d1495b']); ax[0].set_title('Rows per split')
sp.podium_rate.plot.bar(ax=ax[1],color=['#2e86ab','#f6ae2d','#d1495b']); ax[1].set_title('Podium prevalence per split'); savefig('13_split')

## 8. Feature selection evidence (computed on the **training years only**)
Three pieces of evidence: (a) univariate ROC-AUC per feature, (b) correlation/redundancy, (c) **trial-and-error forward selection** by feature group (kept in the notebook): add one group at a time and watch validation PR-AUC.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, precision_recall_curve, roc_curve
uni=[]
for f in FEATURES:
    x=tr[f]; m=x.notna()
    if m.sum()>50 and x[m].nunique()>1:
        a=roc_auc_score(ytr[m.values],x[m]); uni.append((f,FEAT_GROUP_OF[f],a,abs(a-.5)*2,m.mean()))
UNI=pd.DataFrame(uni,columns=['feature','group','auc','strength','non_missing']).sort_values('strength',ascending=False)
plt.figure(figsize=(9,9)); sns.barplot(data=UNI,y='feature',x='strength',hue='group',dodge=False); plt.title('Univariate separation of podium (|2·AUC−1|), train years'); plt.ylabel('')
savefig('14_univariate_strength')
cm=tr[FEATURES].corr(method='spearman'); plt.figure(figsize=(12,10)); sns.heatmap(cm,cmap='coolwarm',center=0,xticklabels=True,yticklabels=True,cbar_kws={'shrink':.6}); plt.title('Spearman correlation among candidate features (train)')
savefig('15_feature_correlation')
hi=[(a,b,cm.loc[a,b]) for i,a in enumerate(cm.columns) for b in cm.columns[i+1:] if abs(cm.loc[a,b])>0.9]; print('Highly redundant pairs (|ρ|>0.9):',[(a,b,round(c,2)) for a,b,c in hi])
fig,ax=plt.subplots(1,3,figsize=(16,4))
for a,(f,bins) in zip(ax,[('grid_filled',[0,1,2,3,4,5,6,8,10,13,16,40]),('form_pos_5',None),('team_form_points_5',None)]):
    d=tr[[f,'podium']].dropna(); d['bin']=pd.cut(d[f],bins) if bins else pd.qcut(d[f],10,duplicates='drop')
    g=d.groupby('bin',observed=True).podium.mean(); a.plot(range(len(g)),g.values,'o-'); a.set_xticks(range(len(g))); a.set_xticklabels([str(i) for i in g.index],rotation=60,fontsize=7); a.set_title(f'Podium rate vs {f}'); a.set_ylabel('podium rate')
savefig('16_feature_effect_on_target')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
class Winsorizer(BaseEstimator,TransformerMixin):
    # clips each column to [q_lo, q_hi] learned on TRAIN only; NaN preserved
    def __init__(self,lo=0.01,hi=0.99): self.lo=lo; self.hi=hi
    def fit(self,X,y=None):
        X=np.asarray(X,float); self.lo_=np.nanquantile(X,self.lo,axis=0); self.hi_=np.nanquantile(X,self.hi,axis=0); return self
    def transform(self,X): return np.clip(np.asarray(X,float),self.lo_,self.hi_)
def make_prep(order=('winsor','impute','scale')):
    steps={'winsor':Winsorizer(),'impute':SimpleImputer(strategy='median',keep_empty_features=True),'scale':StandardScaler()}
    return Pipeline([(o,steps[o]) for o in order])
def best_threshold(y,p):
    pr,rc,th=precision_recall_curve(y,p); f=2*pr*rc/np.clip(pr+rc,1e-12,None); i=np.nanargmax(f[:-1]); return float(th[i])
def metrics(y,p,thr):
    yh=(p>=thr).astype(int)
    return {'ROC-AUC':roc_auc_score(y,p),'PR-AUC':average_precision_score(y,p),'F1':f1_score(y,yh),'Precision':precision_score(y,yh,zero_division=0),'Recall':recall_score(y,yh)}
def build_sk(family,params=None):
    params=params or {}
    if family=='LR':  return Pipeline([('prep',make_prep()),('clf',LogisticRegression(max_iter=4000,**params))])
    if family=='RF':  return Pipeline([('prep',make_prep(('impute',))),('clf',RandomForestClassifier(n_estimators=300,n_jobs=-1,random_state=SEED,**params))])
    if family=='HGB': return Pipeline([('prep',make_prep(('impute',))),('clf',HistGradientBoostingClassifier(max_iter=250,random_state=SEED,**params))])
def val_pr(family,cols,params=None):
    m=build_sk(family,params).fit(Xtr[cols],ytr); return average_precision_score(yva,m.predict_proba(Xva[cols])[:,1])
# ---- Trial-and-error: forward selection by feature group (validation PR-AUC only)
order=['grid_quali','driver_form','constructor_form','standings','experience_context','circuit_history','home']
cols=[];fs=[]
for g in order:
    cols=cols+FEATURE_GROUPS[g]
    fs.append((f'+ {g}',len(cols),val_pr('LR',cols,{'C':1.0}),val_pr('HGB',cols,{'learning_rate':.05,'max_depth':3})))
FS=pd.DataFrame(fs,columns=['step','n_features','LR val PR-AUC','HGB val PR-AUC']); display(FS.round(4))
FS.set_index('step')[['LR val PR-AUC','HGB val PR-AUC']].plot(marker='o',figsize=(9,4)); plt.xticks(rotation=30,ha='right'); plt.title('Forward feature-group trials (validation PR-AUC)'); plt.ylabel('PR-AUC')
savefig('17_forward_selection')

**Decision rule (applied by code in the next cell, on VALIDATION data only — the test set is untouched):**
1. *Group pruning.* For every non-core group, drop it and re-fit the baseline LR and HGB. If removing the group **raises** mean validation PR-AUC by at least `DROP_TOL = 0.005` (≈ the seed-to-seed standard deviation of the NN, 0.006), the group adds noise rather than signal and is **dropped**. `grid_quali` is the core group and is never tested for removal.
2. *Redundancy pruning.* Pairs with |Spearman ρ| > 0.9 are collapsed (keep the feature with the stronger univariate AUC) **only if** that improves validation PR-AUC by ≥ `DROP_TOL`; otherwise the correlated features are kept because they do no measurable harm.
3. Dropped groups are re-added in the ablation (§10) as a cross-check of the decision.

Every kept feature is valid at the pre-race cutoff (table in §6). Dropped features remain in the tables (`CANDIDATES`) so the inference function can still accept them.


In [ ]:
# ---- Evidence-based feature pruning (validation only) ------------------------------------------
CANDIDATES=list(FEATURES); FEATURE_GROUPS_ALL={g:list(fs) for g,fs in FEATURE_GROUPS.items()}
cfg_sel={'LR':{'C':1.0},'HGB':{'learning_rate':.05,'max_depth':3}}
base_v={k:val_pr(k,CANDIDATES,p) for k,p in cfg_sel.items()}
rows=[]
for g,fs in FEATURE_GROUPS_ALL.items():
    if g=='grid_quali': continue
    cols=[f for f in CANDIDATES if f not in fs]; d={k:val_pr(k,cols,p)-base_v[k] for k,p in cfg_sel.items()}
    rows.append((g,len(fs),d['LR'],d['HGB'],float(np.mean(list(d.values())))))
GSEL=pd.DataFrame(rows,columns=['group','n_features','Δ val PR-AUC LR (group removed)','Δ val PR-AUC HGB (group removed)','mean Δ']).sort_values('mean Δ',ascending=False)
DROP_TOL=0.005; DROP_GROUPS=GSEL[GSEL['mean Δ']>=DROP_TOL].group.tolist()
GSEL['decision']=np.where(GSEL.group.isin(DROP_GROUPS),'DROP (removal helps ≥ tol)','keep'); display(GSEL.round(4))
kept=[f for f in CANDIDATES if FEAT_GROUP_OF[f] not in DROP_GROUPS]
# redundancy trial
UNI_S=UNI.set_index('feature').strength; red_drop=set()
for a,b,c in hi:
    if a in red_drop or b in red_drop or a not in kept or b not in kept: continue
    red_drop.add(a if UNI_S.get(a,0)<UNI_S.get(b,0) else b)
cols_red=[f for f in kept if f not in red_drop]
RED=pd.DataFrame([{'set':'groups pruned only','n_features':len(kept),**{f'{k} val PR-AUC':val_pr(k,kept,p) for k,p in cfg_sel.items()}},
                  {'set':'+ redundancy pruned','n_features':len(cols_red),**{f'{k} val PR-AUC':val_pr(k,cols_red,p) for k,p in cfg_sel.items()}}])
gain=float(np.mean([RED.iloc[1][f'{k} val PR-AUC']-RED.iloc[0][f'{k} val PR-AUC'] for k in cfg_sel])); display(RED.round(4))
APPLY_RED=gain>=DROP_TOL
FEATURES=cols_red if APPLY_RED else kept
FEATURE_GROUPS={g:[f for f in fs if f in FEATURES] for g,fs in FEATURE_GROUPS_ALL.items()}; FEATURE_GROUPS={g:fs for g,fs in FEATURE_GROUPS.items() if fs}
DROPPED=[f for f in CANDIDATES if f not in FEATURES]
SEL_NOTE=(f"Groups dropped (validation PR-AUC does not suffer without them, mean Δ ≥ {DROP_TOL}): {DROP_GROUPS or 'none'}. Redundancy pruning "+("APPLIED" if APPLY_RED else "not applied")+f" (mean Δ {gain:+.4f}, tolerance {DROP_TOL}). Final feature set: {len(FEATURES)} of {len(CANDIDATES)} candidates.")
print(SEL_NOTE); ANSWERS['selection']=SEL_NOTE
fig,ax=plt.subplots(figsize=(8,3.5)); GSEL.set_index('group')[['Δ val PR-AUC LR (group removed)','Δ val PR-AUC HGB (group removed)']].plot.barh(ax=ax); ax.axvline(DROP_TOL,c='k',ls='--'); ax.axvline(0,c='k',lw=.8); ax.set_title('Validation PR-AUC change when a group is removed (right of dashed line ⇒ drop)'); savefig('17b_group_pruning')
# ---- is_pit_start diagnostic: grid==0 in Ergast also covers non-starters -> check it is not an outcome proxy
z=tr[tr.grid==0]; print(f'grid==0 rows in train: {len(z)} ({len(z)/len(tr):.1%}); podium rate {z.podium.mean():.3f}; top statuses:',z.status.value_counts().head(5).to_dict())
if 'is_pit_start' in FEATURES:
    cc=[f for f in FEATURES if f!='is_pit_start']; print(f'LR val PR-AUC with is_pit_start {val_pr("LR",FEATURES,{"C":1.0}):.4f} vs without {val_pr("LR",cc,{"C":1.0}):.4f}  (flag is kept only if it does not matter / helps; see limitations)')


## 9. Predictive modeling
All models use **the same rows and the same feature list**, fit on **train**; hyper-parameters and the F1 threshold are chosen on **validation only**; the **test** set is only scored.

| Model | How it works | Main limitations |
|---|---|---|
| **Logistic regression** (baseline) | linear log-odds of podium from standardized features, L2 penalty | no interactions/non-linearity (grid effect is non-linear); sensitive to era drift |
| **Random forest** | many de-correlated deep trees, averaged probabilities | probabilities poorly calibrated; large; no extrapolation |
| **Gradient boosting (HistGB)** | sequential shallow trees fitted to residual errors | can overfit era-specific patterns; needs tuning |
| **Shallow NN (PyTorch, 1–2 hidden layers)** | ReLU MLP trained with BCE, AdamW, dropout, early stopping on validation loss | small tabular data → rarely beats boosting; seed variance; less transparent |

**Metrics.** *Primary: PR-AUC* (podium prevalence ≈ 15%, so ranking positives well matters; ROC-AUC is optimistic under imbalance). *Secondary: ROC-AUC* and *F1* at the validation-tuned threshold. Precision and Recall at that threshold are reported as the remaining two of the five metrics.

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(2)
class TorchMLP(ClassifierMixin,BaseEstimator):
    def __init__(self,hidden=(32,),dropout=0.2,lr=2e-3,weight_decay=1e-4,epochs=200,patience=15,batch_size=256,seed=42):
        self.hidden=hidden; self.dropout=dropout; self.lr=lr; self.weight_decay=weight_decay; self.epochs=epochs; self.patience=patience; self.batch_size=batch_size; self.seed=seed
    def fit(self,X,y,eval_set=None):
        torch.manual_seed(self.seed); g=torch.Generator().manual_seed(self.seed)
        X=torch.tensor(np.asarray(X,np.float32)); y=torch.tensor(np.asarray(y,np.float32))
        layers=[]; d=X.shape[1]
        for h in self.hidden: layers+=[nn.Linear(d,h),nn.ReLU(),nn.Dropout(self.dropout)]; d=h
        layers.append(nn.Linear(d,1)); self.net_=nn.Sequential(*layers)
        opt=torch.optim.AdamW(self.net_.parameters(),lr=self.lr,weight_decay=self.weight_decay); lossf=nn.BCEWithLogitsLoss()
        if eval_set is not None: Xv=torch.tensor(np.asarray(eval_set[0],np.float32)); yv=torch.tensor(np.asarray(eval_set[1],np.float32))
        best=1e9; bad=0; best_state=None; self.history_=[]
        for ep in range(self.epochs):
            self.net_.train(); perm=torch.randperm(len(X),generator=g); tl=0
            for i in range(0,len(X),self.batch_size):
                idx=perm[i:i+self.batch_size]; opt.zero_grad(); l=lossf(self.net_(X[idx]).squeeze(1),y[idx]); l.backward(); opt.step(); tl+=l.item()*len(idx)
            vl=np.nan
            if eval_set is not None:
                self.net_.eval()
                with torch.no_grad(): vl=lossf(self.net_(Xv).squeeze(1),yv).item()
                if vl<best-1e-5: best=vl; bad=0; best_state={k:v.clone() for k,v in self.net_.state_dict().items()}
                else:
                    bad+=1
                    if bad>=self.patience: break
            self.history_.append((tl/len(X),vl))
        if best_state is not None: self.net_.load_state_dict(best_state)
        self.classes_=np.array([0,1]); return self
    def predict_proba(self,X):
        self.net_.eval()
        with torch.no_grad(): p=torch.sigmoid(self.net_(torch.tensor(np.asarray(X,np.float32))).squeeze(1)).numpy()
        return np.c_[1-p,p]
    def predict(self,X): return (self.predict_proba(X)[:,1]>=.5).astype(int)
def fit_nn(cols,cfg,seed=SEED,Xa=None,ya=None,Xv=None,yv=None):
    Xa=Xtr if Xa is None else Xa; ya=ytr if ya is None else ya; Xv=Xva if Xv is None else Xv; yv=yva if yv is None else yv
    prep=make_prep().fit(Xa[cols]); net=TorchMLP(seed=seed,**cfg).fit(prep.transform(Xa[cols]),ya,eval_set=(prep.transform(Xv[cols]),yv))
    return Pipeline([('prep',prep),('clf',net)])
print('torch',torch.__version__)

In [ ]:
RESULTS=[]; PROBS={}; MODELS={}; BEST_PARAMS={}
def evaluate(name,est,cols=FEATURES,family=None):
    P={s:est.predict_proba(X[cols])[:,1] for s,X in [('train',Xtr),('val',Xva),('test',Xte)]}
    thr=best_threshold(yva,P['val'])      # threshold from VALIDATION only
    for s,y in [('train',ytr),('val',yva),('test',yte)]: RESULTS.append({'model':name,'split':s,'threshold':thr,**metrics(y,P[s],thr)})
    PROBS[name]=P; MODELS[name]=est; return thr
# ---------- reference points (simplest approaches)
rule=lambda X:np.clip(1-(X.grid_filled.values-1)/(X.field_size.values),0,1)
Pr={s:rule(X) for s,X in [('train',Xtr),('val',Xva),('test',Xte)]}
for s,y in [('train',ytr),('val',yva),('test',yte)]: RESULTS.append({'model':'Reference: grid-position rule','split':s,'threshold':np.nan,'ROC-AUC':roc_auc_score(y,Pr[s]),'PR-AUC':average_precision_score(y,Pr[s]),**{k:np.nan for k in ['F1','Precision','Recall']}})
# F1 for rule: predict podium iff grid<=3
for s,X,y in [('train',Xtr,ytr),('val',Xva,yva),('test',Xte,yte)]:
    yh=(X.grid_filled.values<=3).astype(int); r=[r for r in RESULTS if r['model']=='Reference: grid-position rule' and r['split']==s][0]; r.update(F1=f1_score(y,yh),Precision=precision_score(y,yh),Recall=recall_score(y,yh),threshold=3)
# ---------- ATTEMPT 1: baseline logistic regression
evaluate('LR baseline (C=1)',build_sk('LR',{'C':1.0}).fit(Xtr[FEATURES],ytr))
# ---------- ATTEMPT 2: LR grid-only (does the extra feature set earn its complexity?)
evaluate('LR grid-only',build_sk('LR').fit(Xtr[FEATURE_GROUPS['grid_quali']],ytr),FEATURE_GROUPS['grid_quali'])
# ---------- ATTEMPT 3: tuned LR (validation PR-AUC)
grid_lr=[{'C':c} for c in [0.01,0.1,1,10]]; sc=[(val_pr('LR',FEATURES,p),p) for p in grid_lr]; BEST_PARAMS['LR']=max(sc,key=lambda t:t[0])[1]
evaluate('LR tuned',build_sk('LR',BEST_PARAMS['LR']).fit(Xtr[FEATURES],ytr)); print('LR search:',[(p['C'],round(s,4)) for s,p in sc])
# ---------- ATTEMPT 4: random forest
grid_rf=[{'max_depth':d,'min_samples_leaf':l} for d in [6,12] for l in [5,20]]; sc=[(val_pr('RF',FEATURES,p),p) for p in grid_rf]; BEST_PARAMS['RF']=max(sc,key=lambda t:t[0])[1]
evaluate('Random forest',build_sk('RF',BEST_PARAMS['RF']).fit(Xtr[FEATURES],ytr)); print('RF search:',[(p,round(s,4)) for s,p in sc])
# ---------- ATTEMPT 5: gradient boosting
grid_hgb=[{'learning_rate':lr,'max_depth':d,'l2_regularization':l2,'min_samples_leaf':40} for lr in [0.03,0.08] for d in [3,5] for l2 in [0.0,1.0]]
sc=[(val_pr('HGB',FEATURES,p),p) for p in grid_hgb]; BEST_PARAMS['HGB']=max(sc,key=lambda t:t[0])[1]
evaluate('Gradient boosting',build_sk('HGB',BEST_PARAMS['HGB']).fit(Xtr[FEATURES],ytr)); print('HGB search:',[(p['learning_rate'],p['max_depth'],p['l2_regularization'],round(s,4)) for s,p in sc])

In [ ]:
# ---------- ATTEMPTS 6+: shallow neural network (1–2 hidden layers); search on VALIDATION PR-AUC
NN_GRID=[{'hidden':h,'dropout':d,'weight_decay':w} for h in [(32,),(64,),(32,16)] for d in [0.1,0.3] for w in [1e-4,1e-2]]
nn_sc=[]
for cfg in NN_GRID:
    m=fit_nn(FEATURES,cfg); nn_sc.append((average_precision_score(yva,m.predict_proba(Xva[FEATURES])[:,1]),cfg))
NN_SEARCH=pd.DataFrame([{'hidden':str(c['hidden']),'dropout':c['dropout'],'weight_decay':c['weight_decay'],'val PR-AUC':s} for s,c in nn_sc]).sort_values('val PR-AUC',ascending=False); display(NN_SEARCH.round(4))
BEST_PARAMS['NN']=max(nn_sc,key=lambda t:t[0])[1]
nn1=max([t for t in nn_sc if len(t[1]['hidden'])==1],key=lambda t:t[0])[1]; nn2=max([t for t in nn_sc if len(t[1]['hidden'])==2],key=lambda t:t[0])[1]
evaluate('Shallow NN (1 hidden layer)',fit_nn(FEATURES,nn1)); evaluate('Shallow NN (2 hidden layers)',fit_nn(FEATURES,nn2))
best_nn_name='Shallow NN (1 hidden layer)' if BEST_PARAMS['NN']==nn1 else 'Shallow NN (2 hidden layers)'
print('Best NN config:',BEST_PARAMS['NN'],'->',best_nn_name)
# training curve of the best NN
net=MODELS[best_nn_name][-1]; h=np.array(net.history_); plt.figure(figsize=(7,4)); plt.plot(h[:,0],label='train loss'); plt.plot(h[:,1],label='validation loss'); plt.legend(); plt.xlabel('epoch'); plt.ylabel('BCE'); plt.title(f'{best_nn_name}: training curve (early stopping on validation loss)')
savefig('18_nn_training_curve')
# seed variance of the chosen NN
sv=[average_precision_score(yva,fit_nn(FEATURES,BEST_PARAMS['NN'],seed=s).predict_proba(Xva[FEATURES])[:,1]) for s in [1,2,3,4,5]]
NN_SEED_STD=float(np.std(sv)); print(f'NN val PR-AUC over 5 seeds: mean {np.mean(sv):.4f} ± {np.std(sv):.4f}')

In [ ]:
RES=pd.DataFrame(RESULTS); metric_cols=['ROC-AUC','PR-AUC','F1','Precision','Recall']
tbl=RES.pivot_table(index='model',columns='split',values=metric_cols)[[(m,s) for m in metric_cols for s in ['train','val','test']]]
display(tbl.round(3))
comp=RES[RES.split.isin(['val','test'])&RES.model.str.contains('LR|Random|Gradient|NN|Reference')]
fig,ax=plt.subplots(1,3,figsize=(17,5))
for a,m in zip(ax,['PR-AUC','ROC-AUC','F1']):
    sns.barplot(data=comp,y='model',x=m,hue='split',ax=a,palette=['#f6ae2d','#d1495b']); a.set_title(m+' (validation vs test)'); a.set_ylabel('')
    if a is not ax[0]: a.get_legend().remove()
savefig('19_model_comparison')
fig,ax=plt.subplots(1,2,figsize=(13,5)); names=[n for n in PROBS if n in ['LR tuned','Random forest','Gradient boosting',best_nn_name]]
for n in names:
    fpr,tpr,_=roc_curve(yte,PROBS[n]['test']); ax[0].plot(fpr,tpr,label=f"{n} ({roc_auc_score(yte,PROBS[n]['test']):.3f})")
    pr,rc,_=precision_recall_curve(yte,PROBS[n]['test']); ax[1].plot(rc,pr,label=f"{n} ({average_precision_score(yte,PROBS[n]['test']):.3f})")
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_title('ROC — unseen test'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
ax[1].axhline(yte.mean(),c='k',ls='--'); ax[1].set_title('Precision–Recall — unseen test'); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].legend(); savefig('20_roc_pr_test')
cand={n:RES[(RES.model==n)&(RES.split=='val')]['PR-AUC'].iloc[0] for n in ['LR tuned','Random forest','Gradient boosting',best_nn_name]}
FINAL_NAME=max(cand,key=cand.get); FINAL=MODELS[FINAL_NAME]; FINAL_THR=RES[(RES.model==FINAL_NAME)&(RES.split=='val')].threshold.iloc[0]
print('Candidates by VALIDATION PR-AUC:',{k:round(v,4) for k,v in cand.items()}); print('FINAL MODEL (selected on validation):',FINAL_NAME,'| threshold',round(FINAL_THR,3))
base_te=RES[(RES.model=='LR baseline (C=1)')&(RES.split=='test')]['PR-AUC'].iloc[0]; fin_te=RES[(RES.model==FINAL_NAME)&(RES.split=='test')]['PR-AUC'].iloc[0]
ANSWERS['model']=f"Final model **{FINAL_NAME}** (best validation PR-AUC). Test PR-AUC {fin_te:.3f} vs baseline LR {base_te:.3f}; grid-rule reference {RES[(RES.model=='Reference: grid-position rule')&(RES.split=='test')]['PR-AUC'].iloc[0]:.3f}."
display(Markdown(ANSWERS['model']))

**Reading the table:** train ≫ validation/test for flexible models (RF, GBM) indicates over-fitting to the training seasons; a small train→val gap for LR/NN indicates under-fitting rather than variance. The F1 threshold is fixed from validation and reused on train and test. The comparison with the grid-only LR and the grid-rule reference shows how much the engineered form/standings features add beyond starting position.

## 10. Experiment: feature-group ablation (drop one group → re-tune threshold → re-train; LR, RF, HGB and NN — RF is the final model; grid_quali split into grid / qualifying / penalty)

In [ ]:
def refit(family,cols,seed=SEED):
    if family=='NN': return fit_nn(cols,BEST_PARAMS['NN'],seed=seed)
    return build_sk(family,BEST_PARAMS[family]).fit(Xtr[cols],ytr)
def score_cols(family,cols,seeds=(1,2,3)):
    out=[]
    for sd in (seeds if family=='NN' else (SEED,)):
        m=refit(family,cols,sd); pv=m.predict_proba(Xva[cols])[:,1]; pt=m.predict_proba(Xte[cols])[:,1]; thr=best_threshold(yva,pv)
        out.append([average_precision_score(yva,pv),roc_auc_score(yva,pv),f1_score(yva,pv>=thr),average_precision_score(yte,pt),roc_auc_score(yte,pt),f1_score(yte,pt>=thr)])
    return np.mean(out,axis=0)
# grid_quali is split so grid, qualifying and the grid-vs-qualifying penalty signal are ablated separately
SPLIT={'grid':['grid_filled','is_pit_start','front_row'],'qualifying':['quali_position','quali_missing','q1_gap_pct','best_gap_pct'],'penalty (grid−quali)':['grid_minus_quali','grid_penalty']}
ABL_GROUPS={g:fs for g,fs in FEATURE_GROUPS.items() if g!='grid_quali'}
for g,fs in SPLIT.items(): ABL_GROUPS[g]=[f for f in fs if f in FEATURES]
ABL_GROUPS={g:fs for g,fs in ABL_GROUPS.items() if fs}
ONLY={'ONLY grid_quali (all 3 sub-groups)':[f for f in FEATURES if f in FEATURE_GROUPS_ALL['grid_quali']],'ONLY grid':ABL_GROUPS.get('grid',[]),'ONLY qualifying (+penalty)':ABL_GROUPS.get('qualifying',[])+ABL_GROUPS.get('penalty (grid−quali)',[])}
ab=[]
for fam in ['LR','RF','HGB','NN']:          # RF is included because it is the final model
    full=score_cols(fam,FEATURES); ab.append((fam,'ALL FEATURES',*full,0,0))
    for g,fs in ABL_GROUPS.items():
        cols=[f for f in FEATURES if f not in fs]; s=score_cols(fam,cols); ab.append((fam,f'− {g}',*s,s[0]-full[0],s[3]-full[3]))
    for lab,cols in ONLY.items():
        if cols: o=score_cols(fam,cols); ab.append((fam,lab,*o,o[0]-full[0],o[3]-full[3]))
    if DROPPED:                              # cross-check of the pruning decision of §8
        o=score_cols(fam,FEATURES+DROPPED); ab.append((fam,'+ re-added pruned features',*o,o[0]-full[0],o[3]-full[3]))
AB=pd.DataFrame(ab,columns=['model','setting','val PR-AUC','val ROC-AUC','val F1','test PR-AUC','test ROC-AUC','test F1','Δ val PR-AUC','Δ test PR-AUC']); display(AB.round(4))
fig,ax=plt.subplots(1,2,figsize=(16,5))
for a,c in zip(ax,['Δ val PR-AUC','Δ test PR-AUC']):
    d=AB[AB.setting.str.startswith('−')]; sns.barplot(data=d,y='setting',x=c,hue='model',ax=a); a.axvline(0,c='k'); a.set_title(c+' when the group is removed (negative = group helps)'); a.set_ylabel('')
    if a is ax[1]: a.get_legend().remove()
savefig('21_ablation')
parts=[]
for fam in AB.model.unique():
    d=AB[(AB.model==fam)&AB.setting.str.startswith('−')].sort_values('Δ val PR-AUC'); parts.append(f"{fam}: most valuable group {d.iloc[0].setting} ({d.iloc[0]['Δ val PR-AUC']:+.3f} val PR-AUC when removed), least valuable {d.iloc[-1].setting} ({d.iloc[-1]['Δ val PR-AUC']:+.3f})")
ANSWERS['ablation']=' | '.join(parts)
display(Markdown('**Ablation summary.** '+ANSWERS['ablation']+' Correlated groups can mask each other (e.g. grid vs standings), so a small drop does not mean a group is useless — see the `ONLY …` rows for the opposite direction and the `+ re-added pruned features` row for a check of the §8 pruning decision.'))

## 11. Experiments: preprocessing order & pipeline-order effects

* **EXP-A — where preprocessing is fitted:** fit imputer/winsorizer/scaler on train only (correct) vs on the whole table before splitting (leaky).
* **EXP-B — operation order:** winsorize→impute→scale vs impute→winsorize→scale vs impute→scale→winsorize (LR).
* **EXP-C — engineer-then-filter vs filter-then-engineer:** build rolling features on full history and then drop pre-start seasons (our choice), or drop first (form features lose their warm-up).
* **EXP-D — random vs chronological split:** shows how much a random split inflates scores.
* **EXP-E — modeling start year:** all seasons vs the qualifying-covered window.

In [ ]:
def fit_score_lr(prep_fit_X,order,cols=FEATURES,Xa=Xtr,ya=ytr,Xv=Xva,yv=yva,Xt=Xte,yt=yte):
    prep=make_prep(order).fit(prep_fit_X[cols]); lr=LogisticRegression(max_iter=4000,C=BEST_PARAMS['LR']['C']).fit(prep.transform(Xa[cols]),ya)
    pv=lr.predict_proba(prep.transform(Xv[cols]))[:,1]; pt=lr.predict_proba(prep.transform(Xt[cols]))[:,1]
    return average_precision_score(yv,pv),average_precision_score(yt,pt),roc_auc_score(yt,pt)
rowsE=[]
# EXP-A
a1=fit_score_lr(Xtr,('winsor','impute','scale')); a2=fit_score_lr(MODEL[FEATURES],('winsor','impute','scale'))
rowsE+= [('A','fit on TRAIN only (correct)',*a1),('A','fit on ALL data (leaky)',*a2)]
# EXP-B
for o in [('winsor','impute','scale'),('impute','winsor','scale'),('impute','scale','winsor')]: rowsE.append(('B',' → '.join(o),*fit_score_lr(Xtr,o)))
EXP=pd.DataFrame(rowsE,columns=['exp','variant','val PR-AUC','test PR-AUC','test ROC-AUC'])
# EXP-C
t0=time.time(); ALT=build_features(BASE[BASE.year>=START_YEAR]); ALT=ALT[ALT.year>=START_YEAR].reset_index(drop=True)
ALT['split']=np.where(ALT.year.isin(TEST_YEARS),'test',np.where(ALT.year.isin(VAL_YEARS),'val','train'))
a,v,t=[ALT[ALT.split==s] for s in ['train','val','test']]
c2=fit_score_lr(a[FEATURES],('winsor','impute','scale'),Xa=a[FEATURES],ya=a.podium.values,Xv=v[FEATURES],yv=v.podium.values,Xt=t[FEATURES],yt=t.podium.values)
EXP=pd.concat([EXP,pd.DataFrame([('C','engineer on full history, then filter (chosen)',*a1),('C','filter first, then engineer',*c2)],columns=EXP.columns)])
# EXP-D (random split on the same rows)
from sklearn.model_selection import train_test_split
idx_tr,idx_rest=train_test_split(np.arange(len(MODEL)),test_size=len(va)+len(te),random_state=SEED,stratify=MODEL.podium)
idx_va,idx_te=train_test_split(idx_rest,test_size=len(te),random_state=SEED)
R=lambda ix:(MODEL.iloc[ix][FEATURES],MODEL.iloc[ix].podium.values)
(Xa,ya),(Xv,yv),(Xt,yt)=R(idx_tr),R(idx_va),R(idx_te)
d2=fit_score_lr(Xa,('winsor','impute','scale'),Xa=Xa,ya=ya,Xv=Xv,yv=yv,Xt=Xt,yt=yt)
EXP=pd.concat([EXP,pd.DataFrame([('D','chronological split (chosen)',*a1),('D','random split (same sizes)',*d2)],columns=EXP.columns)])
# EXP-E: all seasons vs qualifying-covered window
ALLM=ALL.copy(); ALLM['split']=np.where(ALLM.year.isin(TEST_YEARS),'test',np.where(ALLM.year.isin(VAL_YEARS),'val','train'))
a,v,t=[ALLM[ALLM.split==s] for s in ['train','val','test']]
e2=fit_score_lr(a[FEATURES],('winsor','impute','scale'),Xa=a[FEATURES],ya=a.podium.values,Xv=v[FEATURES],yv=v.podium.values,Xt=t[FEATURES],yt=t.podium.values)
EXP=pd.concat([EXP,pd.DataFrame([('E',f'train from {START_YEAR} (chosen)',*a1),('E','train from 1950 (all seasons)',*e2)],columns=EXP.columns)]).reset_index(drop=True)
display(EXP.round(4))
plt.figure(figsize=(10,6)); sns.barplot(data=EXP.assign(label=EXP.exp+': '+EXP.variant),y='label',x='test PR-AUC',color='#2e86ab'); plt.xlim(max(0,EXP['test PR-AUC'].min()-.05),min(1,EXP['test PR-AUC'].max()+.03)); plt.title('Preprocessing / pipeline-order experiments (LR, test PR-AUC)'); plt.ylabel('')
savefig('22_preprocessing_experiments')
ANSWERS['exp']=EXP.round(4).to_string(index=False)
_e=EXP[EXP.exp=='E'].reset_index(drop=True); dv=_e['val PR-AUC'][1]-_e['val PR-AUC'][0]; dt=_e['test PR-AUC'][1]-_e['test PR-AUC'][0]
ANSWERS['expE']=(f"EXP-E: training from 1950 changes validation PR-AUC by {dv:+.4f} and test PR-AUC by {dt:+.4f} versus training from {START_YEAR}. "
 +("The difference is inside seed/selection noise (NN seed std ≈ "+f"{NN_SEED_STD:.4f}) " if abs(dv)<2*NN_SEED_STD else "The gain is larger than seed noise ")
 +f"and the test gain is {'negligible' if abs(dt)<0.005 else 'visible'}. We keep {START_YEAR} because qualifying times are largely absent earlier (those rows rely on median imputation + missing-flags), the pre-{START_YEAR} cars/regulations are far from the 2022-24 test regime, and the data window then matches the task definition ('after qualifying'); we acknowledge that more history did not hurt and may help slightly.")
display(Markdown('**EXP-E decision.** '+ANSWERS['expE']))


In [ ]:
# Effect of the preprocessing steps on feature distributions/ranges (train only)
show=[f for f in ['q1_gap_pct','form_pos_5','team_form_points_5','age','drv_pts_ratio','circ_avg_pos'] if f in FEATURES][:4]; prepX=make_prep().fit(Xtr[FEATURES]); Z=pd.DataFrame(prepX.transform(Xtr[FEATURES]),columns=FEATURES)
fig,ax=plt.subplots(2,len(show),figsize=(4.2*len(show),6),squeeze=False)
for j,f in enumerate(show):
    ax[0,j].hist(Xtr[f].dropna(),bins=40,color='grey'); ax[0,j].set_title(f'{f}\nraw (NaN={Xtr[f].isna().mean():.0%})'); ax[1,j].hist(Z[f],bins=40,color='#2e86ab'); ax[1,j].set_title('winsorized → imputed → scaled')
savefig('23_preprocessing_distribution')
summ=pd.DataFrame({'raw_min':Xtr[show].min(),'raw_max':Xtr[show].max(),'raw_mean':Xtr[show].mean(),'proc_min':Z[show].min(),'proc_max':Z[show].max(),'proc_mean':Z[show].mean(),'proc_std':Z[show].std()}); display(summ.round(3))

## 12. Explainability (XAI) — for the final model chosen on validation
**Global:** permutation importance (drop in test PR-AUC when a feature / feature group is shuffled) and SHAP.  
**Local:** SHAP waterfall and LIME for individual driver-race predictions.

> ⚠️ **Explanations are not causal effects.** A feature being important means the *model relies on it to predict*; it does not prove that the feature *causes* podium finishes. Correlated features (grid, quali position, team form, standings) share credit, and e.g. "team form" is a proxy for car quality, not a lever a driver can pull.

In [ ]:
from sklearn.inspection import permutation_importance
pi=permutation_importance(FINAL,Xte[FEATURES],yte,scoring='average_precision',n_repeats=10,random_state=SEED,n_jobs=1)
PI=pd.DataFrame({'feature':FEATURES,'group':[FEAT_GROUP_OF[f] for f in FEATURES],'mean':pi.importances_mean,'std':pi.importances_std}).sort_values('mean',ascending=False)
top=PI.head(15).iloc[::-1]; plt.figure(figsize=(8,6)); plt.barh(top.feature,top['mean'],xerr=top['std'],color='#2e86ab'); plt.xlabel('drop in test PR-AUC when shuffled'); plt.title(f'Permutation importance — {FINAL_NAME} (test)')
savefig('24_permutation_importance')
# grouped permutation importance (all columns of a group shuffled jointly)
rng=np.random.default_rng(SEED); base_s=average_precision_score(yte,FINAL.predict_proba(Xte[FEATURES])[:,1]); gp=[]
for g,fs in FEATURE_GROUPS.items():
    d=[]
    for _ in range(10):
        Xp=Xte[FEATURES].copy(); perm=rng.permutation(len(Xp)); Xp[fs]=Xp[fs].values[perm]; d.append(base_s-average_precision_score(yte,FINAL.predict_proba(Xp)[:,1]))
    gp.append((g,np.mean(d),np.std(d)))
GP=pd.DataFrame(gp,columns=['group','mean','std']).sort_values('mean'); plt.figure(figsize=(7,4)); plt.barh(GP.group,GP['mean'],xerr=GP['std'],color='#6a994e'); plt.title('Grouped permutation importance (test PR-AUC drop)'); plt.xlabel('drop in PR-AUC')
savefig('25_grouped_permutation')

In [ ]:
import shap
prep=FINAL[:-1]; clf=FINAL[-1]
Tf=lambda X: pd.DataFrame(prep.transform(X[FEATURES]),columns=FEATURES,index=X.index)
bg=Tf(Xtr.sample(min(150,len(Xtr)),random_state=SEED)); S_raw=Xte.sample(min(300,len(Xte)),random_state=SEED); S=Tf(S_raw)
fn=lambda A: clf.predict_proba(pd.DataFrame(A,columns=FEATURES))[:,1]
def make_explainer():
    if isinstance(clf,RandomForestClassifier) or isinstance(clf,HistGradientBoostingClassifier):
        try:
            e=shap.TreeExplainer(clf); e.shap_values(S.iloc[:3]); return ('tree',e)
        except Exception as ex: print('TreeExplainer unavailable ->',type(ex).__name__,'; using model-agnostic explainer')
    elif isinstance(clf,LogisticRegression): return ('linear',shap.LinearExplainer(clf,bg))
    return ('perm',shap.Explainer(fn,bg,algorithm='permutation',seed=SEED))
kind,EXPL=make_explainer(); print('SHAP explainer type:',kind)
def shap_vals(Xraw):
    Z=Tf(Xraw)
    if kind=='perm': e=EXPL(Z); return e.values,float(np.mean(e.base_values))
    v=EXPL.shap_values(Z); b=EXPL.expected_value
    if isinstance(v,list): v=v[1]; b=b[1]
    elif getattr(v,'ndim',2)==3: v=v[:,:,1]; b=b[1]
    return v,float(np.ravel(b)[-1])
SV,SB=shap_vals(S_raw)
plt.figure(); shap.summary_plot(SV,S_raw[FEATURES],feature_names=FEATURES,show=False,max_display=15); plt.title(f'SHAP beeswarm — {FINAL_NAME}'); savefig('26_shap_beeswarm')
plt.figure(); shap.summary_plot(SV,S_raw[FEATURES],feature_names=FEATURES,plot_type='bar',show=False,max_display=15); plt.title('Mean |SHAP|'); savefig('27_shap_bar')
SH=pd.Series(np.abs(SV).mean(0),index=FEATURES).sort_values(ascending=False)
rho,_=stats.spearmanr(PI.set_index('feature').loc[FEATURES,'mean'],SH.loc[FEATURES]); print(f'Spearman agreement between permutation importance and mean|SHAP| ranks: {rho:.2f}')
print('Top-5 by permutation:',PI.feature.head(5).tolist()); print('Top-5 by SHAP       :',SH.index[:5].tolist())
K=10; top_pi=PI.feature.head(K).tolist(); top_sh=SH.index[:K].tolist(); both=[f for f in top_pi if f in top_sh]
only_pi=[f for f in top_pi if f not in top_sh]; only_sh=[f for f in top_sh if f not in top_pi]
# correlated partners explain disagreement: permuting ONE of two near-duplicates leaves the other intact (permutation under-credits both), SHAP splits credit
cm_all=Xte[FEATURES].corr(method='spearman'); pairs=[(a_,b_,cm_all.loc[a_,b_]) for a_ in only_sh for b_ in top_pi if a_!=b_ and abs(cm_all.loc[a_,b_])>0.8]
pair_txt='; '.join(f"{a_}~{b_} (ρ={r_:.2f})" for a_,b_,r_ in pairs[:4]) or 'none above |ρ|=0.8'
top_groups=GP.sort_values('mean',ascending=False)
dir_txt=[]
for f in SH.index[:5]:
    c=np.corrcoef(S_raw[f].fillna(S_raw[f].median()).values,SV[:,FEATURES.index(f)])[0,1]; dir_txt.append(f"{f} ({'higher value → higher podium probability' if c>0 else 'higher value → lower podium probability'})")
ANSWERS['xai']=(f"Permutation importance top-5: {PI.feature.head(5).tolist()}; SHAP top-5: {SH.index[:5].tolist()}. Rank agreement over all features ρ={rho:.2f}; top-{K} overlap = {len(both)}/{K} ({both}). "
 f"Only in permutation top-{K}: {only_pi}; only in SHAP top-{K}: {only_sh}. Likely reason for the disagreement — correlated feature pairs: {pair_txt}. Permuting one member of a pair leaves its partner intact, so permutation importance can under-state both, while SHAP spreads credit between them; grouped permutation (below) is therefore the fairer group-level view. "
 f"Grouped permutation ranks the groups {top_groups.group.tolist()} (PR-AUC drop {[round(x,3) for x in top_groups['mean'].tolist()]}). Direction of the top SHAP features: {'; '.join(dir_txt)}.")
display(Markdown('**Global interpretation.** '+ANSWERS['xai']+' These describe what the fitted model *uses*, not what *causes* podiums.'))


In [ ]:
# Local explanations: (1) a confidently-correct podium prediction, (2) a missed podium (false negative)
pt=PROBS[FINAL_NAME]['test']; tev=te.reset_index(drop=True); drv=T['drivers'].set_index('driverId'); rn=RACES.set_index('raceId')
ok=np.where((yte==1)&(pt>=FINAL_THR))[0]; i1=ok[np.argmax(pt[ok])] if len(ok) else int(np.argmax(pt))
miss=np.where((yte==1)&(pt<FINAL_THR))[0]; i2=miss[np.argmin(pt[miss])] if len(miss) else None
def who(i): r=tev.iloc[i]; return f"{drv.forename[r.driverId]} {drv.surname[r.driverId]} — {int(r.year)} round {int(r['round'])} (grid {int(r.grid_filled)}, finished P{int(r.positionOrder)})  p(podium)={pt[i]:.2f}"
LOCAL_SHAP={}; LOCAL=[('correct podium',i1)]+([('missed podium',i2)] if i2 is not None else [])
for lab,i in LOCAL:
    v,b=shap_vals(tev.iloc[[i]][FEATURES]); LOCAL_SHAP[lab]=(v[0].copy(),b); ex=shap.Explanation(values=v[0],base_values=b,data=tev.iloc[i][FEATURES].values.astype(float),feature_names=FEATURES)
    plt.figure(); shap.plots.waterfall(ex,max_display=12,show=False); plt.title(f'Local SHAP ({lab}): '+who(i),fontsize=9,loc='left'); savefig(f'28_shap_waterfall_{lab.split()[0]}')
    print(lab.upper(),'→',who(i))

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
med=Xtr[FEATURES].median().fillna(0); Xl=Xtr[FEATURES].fillna(med).values.astype(float)
cat_idx=[j for j,f in enumerate(FEATURES) if Xtr[f].nunique()<=2]
lime_exp=LimeTabularExplainer(Xl,feature_names=FEATURES,class_names=['no podium','podium'],categorical_features=cat_idx,mode='classification',discretize_continuous=True,random_state=SEED)
lime_pred=lambda A: FINAL.predict_proba(pd.DataFrame(A,columns=FEATURES))
LIME_OUT={}
for lab,i in LOCAL:
    e=lime_exp.explain_instance(tev.iloc[i][FEATURES].fillna(med).values.astype(float),lime_pred,num_features=10,num_samples=2000); fig=e.as_pyplot_figure(); fig.set_size_inches(8,4.5); plt.title(f'LIME ({lab}): p(podium)={pt[i]:.2f}',fontsize=10)
    savefig(f'29_lime_{lab.split()[0]}'); LIME_OUT[lab]=e.as_list(); print(lab.upper(),'LIME top rules:',[(r,round(w,3)) for r,w in e.as_list()[:5]])
def _feat_of(rule): return next((f for f in sorted(FEATURES,key=len,reverse=True) if f in rule),rule)
LOC_TXT=[]
for lab,i in LOCAL:
    sv,b0=LOCAL_SHAP[lab]; s_=pd.Series(sv,index=FEATURES); up=s_.sort_values(ascending=False).head(3); dn=s_.sort_values().head(3)
    lim=[(_feat_of(r),w) for r,w in LIME_OUT[lab]]; lim_top=[f for f,_ in lim[:5]]; shap_top5=s_.abs().sort_values(ascending=False).head(5).index.tolist(); ov=[f for f in shap_top5 if f in lim_top]
    LOC_TXT.append(f"**{lab.title()}** — {who(i)}. SHAP base value {b0:.2f} (the model's average output on its native scale: log-odds for boosting/linear models, probability for a random forest). SHAP raises the model output via "+', '.join(f"`{k}` ({v:+.3f})" for k,v in up.items())+" and lowers it via "+', '.join(f"`{k}` ({v:+.3f})" for k,v in dn.items())+
        f". LIME's top rules: "+'; '.join(f"{r} ({w:+.3f})" for r,w in LIME_OUT[lab][:4])+f". SHAP/LIME top-5 feature overlap: {len(ov)}/5 {ov}"+(" — the two methods agree on the main drivers." if len(ov)>=3 else " — the methods disagree on the main drivers, which is a warning that the local attribution is unstable (correlated grid/qualifying features share credit)."))
LOC_TXT.append("Race-day events (safety cars, rain, strategy, first-lap incidents) are not available at the pre-race cutoff, so a missed podium such as the one above cannot be explained by any feature. LIME is a local linear approximation and SHAP a Shapley-value attribution; both describe the model, not causal effects.")
display(Markdown('**Local interpretation.**\n\n'+'\n\n'.join(LOC_TXT)))


## 13. Inference function (complete and partially-missing records)

In [ ]:
def predict_podium(record,model=None,threshold=None):
    # record: dict (or list of dicts / DataFrame) with any subset of the candidate pre-race fields. Missing values are allowed.
    model=FINAL if model is None else model; threshold=FINAL_THR if threshold is None else threshold
    df=pd.DataFrame([record]) if isinstance(record,dict) else pd.DataFrame(record)
    unknown=sorted(set(df.columns)-set(CANDIDATES))
    if unknown: print('Ignoring unknown fields (possible leakage / typo):',unknown)
    X=df.reindex(columns=CANDIDATES).astype(float)
    # 1) semantic missing-flags are derived from what was ORIGINALLY absent
    for val,flag in [('quali_position','quali_missing'),('form_pos_5','no_history'),('team_form_pos_5','team_no_history'),('drv_pts_ratio','standings_missing'),('con_pos_pre','con_standings_missing')]:
        X[flag]=np.where(X[flag].isna(),X[val].isna().astype(float),X[flag])
    # 2) coherent defaults for deterministic relationships (instead of letting the imputer invent an inconsistent record)
    X['front_row']=np.where(X['front_row'].isna()&X.grid_filled.notna(),(X.grid_filled<=2).astype(float),X['front_row'])
    X['is_pit_start']=np.where(X['is_pit_start'].isna()&X.grid_filled.notna(),0.0,X['is_pit_start'])
    nq=X.quali_position.isna()&X.grid_filled.notna()                      # no qualifying info: assume no penalty, i.e. quali position == grid slot
    X['quali_position']=np.where(nq,X.grid_filled,X.quali_position); X['grid_minus_quali']=np.where(X.grid_minus_quali.isna()&X.grid_filled.notna(),X.grid_filled-X.quali_position,X.grid_minus_quali)
    X['grid_penalty']=np.where(X.grid_penalty.isna()&X.grid_minus_quali.notna(),(X.grid_minus_quali>0).astype(float),X.grid_penalty)
    n_miss=X[FEATURES].isna().sum(axis=1).values
    p=model.predict_proba(X[FEATURES])[:,1]        # remaining gaps are median-imputed by the training-set imputer inside the pipeline
    return pd.DataFrame({'podium_probability':p.round(4),'predicted_podium':(p>=threshold).astype(int),'n_features_missing':n_miss,'input_quality':np.where(n_miss==0,'complete','partial: gaps median-imputed'),'threshold':round(threshold,3)})
row=tev.iloc[i1]; complete={f:row[f] for f in CANDIDATES}
partial={f:v for f,v in complete.items() if FEATURE_GROUPS and FEAT_GROUP_OF[f] not in ('driver_form','constructor_form','standings') and f not in ('quali_position','q1_gap_pct','best_gap_pct','grid_minus_quali')}
print('COMPLETE record  ->'); display(predict_podium(complete))
print('PARTIAL record (only grid, context, circuit history, home given; qualifying times, form and standings missing) ->'); display(predict_podium(partial))
print('MINIMAL record (grid only) ->'); display(predict_podium({'grid_filled':1.0}))
print('Batch of two ->'); display(predict_podium([complete,{'grid_filled':18.0,'field_size':20.0}]))
_pc=predict_podium(complete).podium_probability[0]; _pp=predict_podium(partial).podium_probability[0]; _pm=predict_podium({'grid_filled':1.0}).podium_probability[0]
INF_NOTE=(f"Inference behaviour: the complete record scores p={_pc:.2f}; the same record without qualifying gaps, form and standings scores p={_pp:.2f}; a record with only 'grid_filled=1' scores p={_pm:.2f}. "
 "Probabilities fall as inputs disappear because remaining gaps are filled with training-set medians (a mid-field driver/team), so a partial record is a weaker, more conservative prediction — not a like-for-like one. "
 "Deterministic fields (front-row flag, pit-start flag, quali=grid / no penalty) are derived rather than imputed. Treat 'partial' outputs as low-confidence and prefer a missing-data-aware workflow (supply form/standings whenever available).")
display(Markdown('**'+INF_NOTE.split(':')[0]+'.**'+INF_NOTE[len(INF_NOTE.split(':')[0])+1:])); ANSWERS['inference']=INF_NOTE


## 14. Analytical report (auto-generated from this run) and checklist

In [ ]:
def md_table(df):
    d=df.copy(); cols=list(d.columns); out=['| '+' | '.join(map(str,cols))+' |','|'+'|'.join(['---']*len(cols))+'|']
    for _,r in d.iterrows(): out.append('| '+' | '.join(f'{v:.3f}' if isinstance(v,(float,np.floating)) else str(v) for v in r.values)+' |')
    return '\n'.join(out)
mt=RES[RES.split.isin(['val','test'])&RES.model.str.contains('LR|Random|Gradient|NN|Reference')].pivot_table(index='model',columns='split',values=['PR-AUC','ROC-AUC','F1','Precision','Recall'])
mt.columns=[f'{m} ({s})' for m,s in mt.columns]; mt=mt.reset_index()
feat_rows=pd.DataFrame([(g,', '.join(fs)) for g,fs in FEATURE_GROUPS.items()],columns=['group','features'])

# ======================= PDF builder (reportlab; works on Kaggle with no system packages) =======================
import html as _html, textwrap as _tw, re as _re
try: import reportlab
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q','reportlab'])
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, KeepTogether, Preformatted, PageBreak, HRFlowable
import matplotlib as _mpl
_fd=Path(_mpl.get_data_path())/'fonts'/'ttf'                      # DejaVu ships with matplotlib -> full Unicode (ρ, Δ, →, ≥, −, ≫ ...)
for _n,_f in [('DJ','DejaVuSans.ttf'),('DJ-B','DejaVuSans-Bold.ttf'),('DJ-I','DejaVuSans-Oblique.ttf'),('DJ-BI','DejaVuSans-BoldOblique.ttf'),('DJM','DejaVuSansMono.ttf')]:
    pdfmetrics.registerFont(TTFont(_n,str(_fd/_f)))
pdfmetrics.registerFontFamily('DJ',normal='DJ',bold='DJ-B',italic='DJ-I',boldItalic='DJ-BI')
_ST={'body':ParagraphStyle('body',fontName='DJ',fontSize=8.8,leading=12.6,spaceAfter=5,alignment=TA_LEFT),
     'title':ParagraphStyle('title',fontName='DJ-B',fontSize=20,leading=25,spaceAfter=6,textColor=colors.HexColor('#12355b')),
     'h1':ParagraphStyle('h1',fontName='DJ-B',fontSize=13.5,leading=17,spaceBefore=12,spaceAfter=5,textColor=colors.HexColor('#12355b')),
     'h2':ParagraphStyle('h2',fontName='DJ-B',fontSize=10.5,leading=14,spaceBefore=8,spaceAfter=4,textColor=colors.HexColor('#1f5f8b')),
     'bullet':ParagraphStyle('bullet',fontName='DJ',fontSize=8.8,leading=12.6,leftIndent=14,bulletIndent=3,spaceAfter=3),
     'cell':ParagraphStyle('cell',fontName='DJ',fontSize=6.8,leading=8.6),'cellh':ParagraphStyle('cellh',fontName='DJ-B',fontSize=6.8,leading=8.6,textColor=colors.white),
     'cap':ParagraphStyle('cap',fontName='DJ-I',fontSize=7.6,leading=10,spaceAfter=9,textColor=colors.HexColor('#444444'),alignment=1),
     'code':ParagraphStyle('code',fontName='DJM',fontSize=5.6,leading=7.0,backColor=colors.HexColor('#f4f5f7'),borderPadding=4,spaceAfter=8)}
def _inl(t):
    t=_html.escape(t,quote=False)
    t=_re.sub(r'`([^`]+)`',r'<font name="DJM" size="7.8" backColor="#eef0f3">\1</font>',t)
    t=_re.sub(r'\*\*(.+?)\*\*',r'<b>\1</b>',t); t=_re.sub(r'(?<![\*\w])\*(?!\s)([^*\n]+?)\*(?![\*\w])',r'<i>\1</i>',t)
    return t
def _table(rows):
    cells=[[c.strip() for c in r.strip().strip('|').split('|')] for r in rows if not _re.fullmatch(r'[\s|:\-]+',r)]
    n=len(cells[0]); W=letter[0]-1.4*inch
    L=[max(6,min(60,max(len(r[j]) if j<len(r) else 0 for r in cells))) for j in range(n)]; cw=[W*l/sum(L) for l in L]
    data=[[Paragraph(_inl(c),_ST['cellh' if r==0 else 'cell']) for c in row+['']*(n-len(row))] for r,row in enumerate(cells)]
    t=Table(data,colWidths=cw,repeatRows=1)
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#12355b')),('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#f3f6fa')]),
        ('GRID',(0,0),(-1,-1),0.25,colors.HexColor('#c8d0da')),('VALIGN',(0,0),(-1,-1),'TOP'),('LEFTPADDING',(0,0),(-1,-1),3),('RIGHTPADDING',(0,0),(-1,-1),3),('TOPPADDING',(0,0),(-1,-1),2),('BOTTOMPADDING',(0,0),(-1,-1),2)]))
    return [t,Spacer(1,7)]
_FIGN=[0]; _USED=set()
def _figure(name):
    p=FIG/f'{name}.png'
    if not p.exists(): return [Paragraph(f'<i>[figure {name} not found]</i>',_ST['cap'])]
    from reportlab.lib.utils import ImageReader
    w,h=ImageReader(str(p)).getSize(); maxw=letter[0]-1.4*inch; maxh=4.6*inch; k=min(maxw/w,maxh/h,1.0*maxw/w)
    _FIGN[0]+=1; _USED.add(name); cap=name.split('_',1)[1].replace('_',' ') if '_' in name else name
    return [KeepTogether([RLImage(str(p),width=w*k,height=h*k),Paragraph(f'Figure {_FIGN[0]} — {_html.escape(cap)}  <font size="6.5" color="#888888">({name}.png)</font>',_ST['cap'])])]
def build_pdf(md_text,pdf_path,title='F1 Podium Prediction — Analytical Report'):
    _FIGN[0]=0; _USED.clear(); story=[]; para=[]; bullets=[]; tbl=[]; code=None
    def flush_para():
        if para: story.append(Paragraph(_inl(' '.join(x.strip() for x in para)),_ST['body'])); para.clear()
    for raw in md_text.split('\n'):
        ln=raw.rstrip()
        if code is not None:
            if ln.strip().startswith('```'):
                body='\n'.join(w for l in code for w in (_tw.wrap(l,width=165,subsequent_indent='    ',drop_whitespace=False) or ['']))
                story.append(Preformatted(body,_ST['code'])); code=None
            else: code.append(ln)
            continue
        if ln.strip().startswith('```'): flush_para(); code=[]; continue
        if tbl and not ln.startswith('|'): story.extend(_table(tbl)); tbl=[]
        if ln.startswith('|'): flush_para(); tbl.append(ln); continue
        m=_re.fullmatch(r'\s*!\[([^\]]*)\]\(([^)]+)\)\s*',ln)
        if m: flush_para(); story.extend(_figure(Path(m.group(2)).stem)); continue
        if not ln.strip(): flush_para(); continue
        if ln.startswith('# '): flush_para(); story.append(Paragraph(_inl(ln[2:]),_ST['title'])); story.append(HRFlowable(width='100%',color=colors.HexColor('#12355b'),thickness=1.2,spaceAfter=6)); continue
        if ln.startswith('## '): flush_para(); story.append(Paragraph(_inl(ln[3:]),_ST['h1'])); continue
        if ln.startswith('### '): flush_para(); story.append(Paragraph(_inl(ln[4:]),_ST['h2'])); continue
        if _re.match(r'^[\*\-] ',ln): flush_para(); story.append(Paragraph(_inl(ln[2:]),_ST['bullet'],bulletText='•')); continue
        para.append(ln)
    flush_para()
    if tbl: story.extend(_table(tbl))
    left=sorted(p.stem for p in FIG.glob('*.png') if p.stem not in _USED)      # safety net: any figure not placed above goes in an appendix
    if left:
        story.append(PageBreak()); story.append(Paragraph('Appendix — remaining figures',_ST['h1']))
        for n_ in left: story.extend(_figure(n_))
    def _foot(c,d):
        c.saveState(); c.setFont('DJ',7); c.setFillColor(colors.HexColor('#777777'))
        c.drawString(0.7*inch,0.45*inch,title); c.drawRightString(letter[0]-0.7*inch,0.45*inch,f'Page {d.page}'); c.restoreState()
    SimpleDocTemplate(str(pdf_path),pagesize=letter,leftMargin=0.7*inch,rightMargin=0.7*inch,topMargin=0.7*inch,bottomMargin=0.75*inch,title=title,author='F1 podium prediction notebook').build(story,onFirstPage=_foot,onLaterPages=_foot)
    return _FIGN[0]

_te=lambda m,col='PR-AUC': RES[(RES.model==m)&(RES.split=='test')][col].iloc[0]
_d=_te(FINAL_NAME)-_te('LR baseline (C=1)'); _nnname=best_nn_name
MODEL_DISC=(f"**Model comparison.** Final model **{FINAL_NAME}** (best validation PR-AUC) reaches test PR-AUC {_te(FINAL_NAME):.3f} versus {_te('LR baseline (C=1)'):.3f} for the logistic-regression baseline (Δ={_d:+.3f}) and {_te(_nnname):.3f} for the shallow NN; the grid-position rule alone reaches {_te('Reference: grid-position rule'):.3f}. "
 f"The NN's validation PR-AUC varies by ±{NN_SEED_STD:.3f} across seeds, so a gap of {abs(_d):.3f} between FINAL and the baseline is "+("within" if abs(_d)<2*NN_SEED_STD else "above")+" seed-level noise: "+("the extra complexity is NOT clearly earning its keep over the baseline; most of the signal is in the grid/qualifying group (see ablation)." if abs(_d)<2*NN_SEED_STD else "the more complex model gives a real gain over the baseline.")+
 " Tree ensembles over-fit the training seasons more (train PR-AUC ≫ validation), while LR and the NN show a small train→val gap.")
LIM_EXTRA=(f"* **Feature pruning** is decided on validation only: {ANSWERS['selection']}\n* **`is_pit_start`**: in Ergast `grid=0` also covers non-starters, so the flag can carry a little outcome information; see the diagnostic in §8.\n* **Partial records:** {ANSWERS['inference']}")
FIGS=lambda *n:'  \n'.join(f'![{x}](figures/{x}.png)' for x in n)
REPORT=f'''# F1 Podium Prediction — Analytical Report

*Auto-generated by the notebook from the executed run. Figures are in `./figures/`.*

## 1. Data & cleaning
* 14 Ergast tables loaded as raw strings; `\\N` sentinels → NaN, dates/clock times/lap-times typed (lap times → seconds). Raw CSVs verified unchanged by SHA-256.
* PK/FK audit: see tables in notebook §3 (`figures/04_fk_orphans.png`).
* Grain: {ANSWERS['dups']}. Rule: prefer grid>0 row, then best positionOrder.
* Modeling window starts in **{START_YEAR}** (first season after which qualifying coverage stays ≥ 90%; see the EDA figures below).

{FIGS('01_sentinels_before','02_missing_before_after','03_laptime_semantic_typing','04_fk_orphans','05_duplicates_by_decade','06_eda_after_cleaning','06b_eda_target','07_positionorder_before_after')}

## 2. Data-engineering answers
**Q1 (front row → podium by circuit).** {ANSWERS['Q1']}  
{FIGS('08_q1_top_circuits','09_q1_era_change')}

**Q2 (home advantage).** {ANSWERS['Q2']}  
{FIGS('10_q2_home_vs_away','11_q2_forest')}

**Q3 (mechanical retirements).** {ANSWERS['Q3']}  
{FIGS('12_q3_mech_rate')}

## 3. Features and why they are valid at the pre-race cutoff
{md_table(feat_rows)}

{FIGS('15_feature_correlation','16_feature_effect_on_target')}

Form features use `shift(1)` before rolling (past races only); standings come from the previous round of the same season; grid/qualifying are final before the start. Verified by the executable leakage tests (§6.1: manual recompute, current-race perturbation, standings-source ordering).

## 3b. Feature selection
{ANSWERS['selection']}

{FIGS('14_univariate_strength','17_forward_selection','17b_group_pruning')}

## 4. Split
Chronological: train {tr.year.min()}–{tr.year.max()}, validation {VAL_YEARS}, test {TEST_YEARS}. Hyper-parameters and F1 threshold from validation only.

{FIGS('13_split')}

## 5. Models (primary metric PR-AUC; secondary ROC-AUC and F1)
{md_table(mt)}

{ANSWERS['model']}

{MODEL_DISC}

{FIGS('19_model_comparison','20_roc_pr_test','18_nn_training_curve')}

## 6. Ablation and experiments
{ANSWERS['ablation']}

```
{ANSWERS['exp']}
```
{ANSWERS['expE']}

{FIGS('21_ablation','22_preprocessing_experiments','23_preprocessing_distribution')}

## 7. Explainability
{ANSWERS['xai']}  
**These are model-reliance explanations, not causal effects.**  
{FIGS('24_permutation_importance','25_grouped_permutation','26_shap_beeswarm','27_shap_bar','28_shap_waterfall_correct','28_shap_waterfall_missed','29_lime_correct','29_lime_missed')}

## 8. Known limitations
* **Qualifying-coverage drift:** qualifying data are incomplete before ~{START_YEAR}; earlier seasons are excluded and EXP-E shows the effect of including them.
* **Era sensitivity / distribution shift:** regulation changes (2014 hybrid, 2022 ground effect) alter what predicts podiums; the test seasons are the newest regime.
* Podium prevalence is ≈3/field size and varies by era; thresholds tuned on validation may not transfer.
* Small per-circuit and per-constructor samples → wide intervals (Q1, Q3); rebrand lineage mapping is manual.
* Home flag relies on a manual nationality→country map; constructor strength is not controlled in Q2.
* Mechanical-retirement status classification is keyword-based and ambiguous for statuses like "Retired"/tyre damage (sensitivity reported).
* Race-day information (weather, safety cars, strategy, sprint results) is not available/used.
{LIM_EXTRA}
* Explanations describe model behavior, not causation.
'''
Path('f1_podium_report.md').write_text(REPORT,encoding='utf-8')
PDF_PATH=Path('f1_podium_report.pdf'); n_figs=build_pdf(REPORT,PDF_PATH)
display(Markdown(REPORT)); print(f'Report written to {Path("f1_podium_report.md").resolve()} and {PDF_PATH.resolve()} ({PDF_PATH.stat().st_size/1e6:.1f} MB, {n_figs} figures embedded)')
try:
    from IPython.display import FileLink; display(FileLink(str(PDF_PATH)))      # clickable download link in Jupyter; on Kaggle the file also appears under Output
except Exception: pass

## 15. Final results summary (printed + saved to ./results/)

In [ ]:
print('='*90); print('FINAL RESULTS SUMMARY'); print('='*90)
print('\nSTART YEAR:',START_YEAR,'| TRAIN',int(tr.year.min()),'-',int(tr.year.max()),'| VAL',VAL_YEARS,'| TEST',TEST_YEARS,'| rows',len(tr),len(va),len(te))
for k in ['dups','Q1','Q2','Q3','selection','model','ablation','expE','xai','inference']: print(f'\n[{k}]',ANSWERS[k].replace('**',''))
print('\nMODEL COMPARISON (train/val/test; threshold from validation):'); print(tbl.round(3).to_string())
print('\nFINAL MODEL:',FINAL_NAME,'| threshold',round(FINAL_THR,3))
print('\nABLATION:'); print(AB.round(4).to_string(index=False))
print('\nPREPROCESSING / PIPELINE-ORDER EXPERIMENTS:'); print(EXP.round(4).to_string(index=False))
print('\nTOP-10 PERMUTATION IMPORTANCE:'); print(PI.head(10).round(4).to_string(index=False))
print('\nTOP-10 MEAN|SHAP|:'); print(SH.head(10).round(4).to_string())
for name,df in {'model_comparison':tbl,'ablation':AB,'experiments':EXP,'permutation_importance':PI,'grouped_permutation':GP,'forward_selection':FS,'group_pruning':GSEL,'nn_search':NN_SEARCH,'q2_models':Q2,'q3_mech_rates':P,'fk_audit':FKR,'pk_audit':PKR,'sentinel_audit':SENT}.items():
    df.to_csv(RESULTS_DIR/f'{name}.csv')
MODEL.to_csv(RESULTS_DIR/'modeling_table.csv',index=False)
print('\nSaved tables ->',sorted(p.name for p in RESULTS_DIR.iterdir())); print('Figures ->',len(list(FIG.glob('*.png'))),'files in ./figures/'); print('Report -> f1_podium_report.md')